In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style, but we will draw our own text blocks
hep.style.use(hep.style.ROOT)

def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights**2)
    return counts, np.sqrt(variances)

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    # We use ax.text with transform=ax.transAxes to position it relative to the axes 
    # (0,1 is top left). We add a little padding so it doesn't touch the border.
    
    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top')
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 3. Reaction and Energy
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 4. Luminosity and sqrt(s)
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')

def main():
    filename ="~/merged_elecs_test_tuples.root" 
    #filename = "~/AnalysisOut_pythia6_428-1_0_NC_noRad_ep_10x130_q2_1to10_ab_run009_0252_eicrecon_edm4eic.root";
    #filename = "~/AnalysisOut_pythia6_428-1_0_NC_noRad_ep_10x130_q2_10to100_ab_run009_0374_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_pythia6_428-1_0_NC_noRad_ep_10x130_q2_100to1000_ab_run009_0336_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_EpIC1_1_6-1_0_DDVCS_18x275_q2_0_10_edecay_hminus_abconv_run0_0282_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_EpIC1_1_6-1_0_DDVCS_18x275_q2_0_10_edecay_hplus_abconv_run0_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_pythia6-eic_1_2_0_ep_noradcor_10x100_q2_1_10_run020_ab_0055_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_pythia8_306-1_1_D0_10x100_hiDiv_q2_100_run10_0239_eicrecon_edm4eic.root"
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    # =========================================================================
    # INCLUSIVE PHYSICS SELECTION
    # =========================================================================
    df_signal = df.Filter("reco_mass >= 8.0 && reco_mass <= 12.0") \
                  .Filter("abs(sigma_epz - 36.0) <= 3.6") \
                  .Filter("reco_W >= 0.0")
    #df_signal = df.Filter("reco_W >= 0.0") 
    branches = [
        "reco_mass", "reco_pT", "reco_y", "reco_W", "reco_Q2", "reco_t_proj", "sigma_epz",
        "reco_ePlus_pT", "reco_ePlus_eta", 
        "reco_decay_eMinus_pT", "reco_decay_eMinus_eta",
        "reco_scat_eMinus_E", "reco_scat_eMinus_eta",
        "reco_nTracks_Central", "reco_nTracks_Tagger",
        "weight", "weight_proj"
    ]
    
    print("Pulling inclusive events from RDataFrame into NumPy...")
    data = df_signal.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE PLOT DASHBOARD
    # =========================================================================
    plot_configs = [
        {"var": "reco_mass", "bins": np.linspace(8, 12, 80), "xlabel": r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", "logy": True},
        {"var": "reco_pT", "bins": np.linspace(0, 2.0, 40), "xlabel": r"Reco $\Upsilon$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_y", "bins": np.linspace(-4, 4, 40), "xlabel": r"Reco $\Upsilon$ Rapidity $y$", "logy": False},
        {"var": "reco_W", "bins": np.linspace(20, 150, 65), "xlabel": r"Reco $W_{\gamma p}$ [GeV]", "logy": False},
        
        {"var": "reco_Q2", "bins": np.logspace(-5, 2, 50), "xlabel": r"Reco $Q^2$ [GeV$^2$]", "logy": False, "logx": True},
        {"var": "reco_t_proj", "bins": np.linspace(0, 1.5, 30), "xlabel": r"Projective $|t|$ [GeV$^2$]", "logy": True},
        {"var": "sigma_epz", "bins": np.linspace(25, 45, 60), "xlabel": r"Exclusivity $\Sigma(E-P_z)$ [GeV]", "logy": False},
        
        {"var": "reco_ePlus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^+$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_ePlus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^+$ $\eta$", "logy": False},
        {"var": "reco_decay_eMinus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^-$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_decay_eMinus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^-$ $\eta$", "logy": False},
        
        {"var": "reco_scat_eMinus_E", "bins": np.linspace(0, 25, 50), "xlabel": r"Scattered $e^-$ Energy [GeV]", "logy": False},
        {"var": "reco_scat_eMinus_eta", "bins": np.linspace(-10, 4, 70), "xlabel": r"Scattered $e^-$ $\eta$", "logy": True},
        
        {"var": "reco_nTracks_Central", "bins": np.arange(-0.5, 15.5, 1.0), "xlabel": r"Central Tracks (Count)", "logy": True, "discrete": True},
        {"var": "reco_nTracks_Tagger", "bins": np.arange(-0.5, 5.5, 1.0), "xlabel": r"Low-$Q^2$ Tagger Tracks (Count)", "logy": True, "discrete": True}
    ]

    fig, axs = plt.subplots(4, 4, figsize=(24, 20))
    axs = axs.flatten()

    for i, cfg in enumerate(plot_configs):
        ax = axs[i]
        
        var_name = cfg["var"]
        arr = data[var_name]
        
        wgt = data["weight_proj"] if var_name == "reco_t_proj" else data["weight"]
        
        valid_mask = arr > -900.0
        
        plot_arr = arr[valid_mask]
        plot_wgt = wgt[valid_mask]

        if len(plot_arr) > 0:
            counts, errs = make_hist(plot_arr, cfg["bins"], plot_wgt)
            
            hep.histplot(counts, bins=cfg["bins"], yerr=errs, histtype='step', color='black', ax=ax)
            hep.histplot(counts, bins=cfg["bins"], histtype='fill', color='dodgerblue', alpha=0.4, ax=ax)

        ax.set_xlabel(cfg["xlabel"])
        # Changed y-axis label to match ePIC standard (N_evts)
        ax.set_ylabel(r"$N_{evts}$")
        
        # Add the ePIC labels to every plot!
        draw_epic_labels(ax)
        
        if cfg.get("discrete", False):
            tick_positions = cfg["bins"][:-1] + 0.5
            ax.set_xticks(tick_positions)
        
        # Since we added text to the top left, we MUST expand the y-axis limit 
        # heavily so the data doesn't draw over our new text.
        if cfg.get("logy", False):
            ax.set_yscale('log')
            ax.set_ylim(bottom=0.5)
            if np.max(counts) > 0:
                ax.set_ylim(top=np.max(counts) * 100) # Give 2 orders of magnitude space for text
        else:
            if len(plot_arr) > 0 and np.max(counts) > 0:
                ax.set_ylim(0, np.max(counts) * 1.5) # Expand linear space by 50%
                
        if cfg.get("logx", False):
            ax.set_xscale('log')

    # The 16th panel
    axs[15].axis('off')
    axs[15].text(0.5, 0.6, r"$\Upsilon(nS) \rightarrow e^+e^-$", ha='center', va='center', fontsize=24, fontweight='bold')
    axs[15].text(0.5, 0.4, "Exclusive Reconstructed Kinematics\n" + r"$10$ fb$^{-1}$ Expected Yield", ha='center', va='center', fontsize=18)

    fig.tight_layout()
    fig.savefig("Upsilon_Inclusive_Dashboard.png", dpi=300)
    print("Saved Upsilon_Inclusive_Dashboard.png")

if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style, but we will draw our own text blocks
hep.style.use(hep.style.ROOT)

# -------------------------------------------------------------------------
# Weight convention
# -------------------------------------------------------------------------
# The eSTARlight tuple is generated as Upsilon(nS) -> e+e- and the cross
# sections below are the visible e+e- cross sections:
#
#   sigma_visible = sigma(ep -> e p Upsilon(nS)) * BR(Upsilon(nS) -> e+e-)
#
# Therefore, for expected reconstructed e+e- yields, DO NOT apply a BR
# correction.  The optional normalization below only rescales each truth
# state to its target visible cross section.
EXPECTED_LUMI_INVFB = 10.0

TARGET_VISIBLE_XSEC_FB = {
    "1S": 327.584,
    "2S": 84.780,
    "3S": 68.929,
}

M_Y1S_PDG = 9.46030
M_Y2S_PDG = 10.02326
M_Y3S_PDG = 10.3552
TRUTH_STATE_MASS_WINDOW = 0.20

# Options:
#   "raw" keeps the tuple weights exactly as stored.
#   "target_visible_xsec" rescales each truth state so sum(weight)/L matches
#   the visible e+e- cross sections above.
STATE_NORM_MODE = "raw"

def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights**2)
    return counts, np.sqrt(variances)

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    # We use ax.text with transform=ax.transAxes to position it relative to the axes 
    # (0,1 is top left). We add a little padding so it doesn't touch the border.
    
    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top')
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 3. Reaction and Energy
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 4. Luminosity and sqrt(s)
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')


def _truth_state_filter(mass, window=TRUTH_STATE_MASS_WINDOW):
    return f"true_mass >= {mass - window:.12g} && true_mass <= {mass + window:.12g}"

def add_visible_weight_columns(df):
    """Define visible-yield weights for the dashboard.

    weight_visible      = weight      * state_norm
    weight_proj_visible = weight_proj * state_norm

    No branching-ratio correction is applied, because the tuple already
    represents Upsilon(nS) -> e+e- and the target cross sections are visible
    e+e- cross sections.
    """
    scale_1s = 1.0
    scale_2s = 1.0
    scale_3s = 1.0

    if STATE_NORM_MODE == "target_visible_xsec":
        sumw_1s = df.Filter(_truth_state_filter(M_Y1S_PDG)).Sum("weight").GetValue()
        sumw_2s = df.Filter(_truth_state_filter(M_Y2S_PDG)).Sum("weight").GetValue()
        sumw_3s = df.Filter(_truth_state_filter(M_Y3S_PDG)).Sum("weight").GetValue()

        target_1s = TARGET_VISIBLE_XSEC_FB["1S"] * EXPECTED_LUMI_INVFB
        target_2s = TARGET_VISIBLE_XSEC_FB["2S"] * EXPECTED_LUMI_INVFB
        target_3s = TARGET_VISIBLE_XSEC_FB["3S"] * EXPECTED_LUMI_INVFB

        scale_1s = target_1s / sumw_1s if sumw_1s > 0 else 1.0
        scale_2s = target_2s / sumw_2s if sumw_2s > 0 else 1.0
        scale_3s = target_3s / sumw_3s if sumw_3s > 0 else 1.0

        print("\n--- Visible state-normalization factors ---")
        print(f"1S: raw visible xsec = {sumw_1s / EXPECTED_LUMI_INVFB:.3f} fb, target = {TARGET_VISIBLE_XSEC_FB['1S']:.3f} fb, scale = {scale_1s:.6g}")
        print(f"2S: raw visible xsec = {sumw_2s / EXPECTED_LUMI_INVFB:.3f} fb, target = {TARGET_VISIBLE_XSEC_FB['2S']:.3f} fb, scale = {scale_2s:.6g}")
        print(f"3S: raw visible xsec = {sumw_3s / EXPECTED_LUMI_INVFB:.3f} fb, target = {TARGET_VISIBLE_XSEC_FB['3S']:.3f} fb, scale = {scale_3s:.6g}")
        print("No BR correction is applied for reconstructed e+e- yields.")
        print("-------------------------------------------\n")
    elif STATE_NORM_MODE != "raw":
        raise ValueError(f"Unknown STATE_NORM_MODE: {STATE_NORM_MODE}")

    state_norm_expr = (
        f"(fabs(true_mass - {M_Y1S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? ({scale_1s}) : "
        f"(fabs(true_mass - {M_Y2S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? ({scale_2s}) : "
        f"(fabs(true_mass - {M_Y3S_PDG}) < {TRUTH_STATE_MASS_WINDOW}) ? ({scale_3s}) : "
        "1.0"
    )

    return (
        df.Define("state_norm", state_norm_expr)
          .Define("weight_visible", "weight * state_norm")
          .Define("weight_proj_visible", "weight_proj * state_norm")
    )


def main():
    filename ="~/merged_elecs_test_tuples_fix2.root" 
    #filename = "~/AnalysisOut_pythia6_428-1_0_NC_noRad_ep_10x130_q2_1to10_ab_run009_0252_eicrecon_edm4eic.root";
    #filename = "~/AnalysisOut_pythia6_428-1_0_NC_noRad_ep_10x130_q2_10to100_ab_run009_0374_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_pythia6_428-1_0_NC_noRad_ep_10x130_q2_100to1000_ab_run009_0336_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_EpIC1_1_6-1_0_DDVCS_18x275_q2_0_10_edecay_hminus_abconv_run0_0282_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_EpIC1_1_6-1_0_DDVCS_18x275_q2_0_10_edecay_hplus_abconv_run0_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_pythia6-eic_1_2_0_ep_noradcor_10x100_q2_1_10_run020_ab_0055_eicrecon_edm4eic.root"
    #filename = "~/AnalysisOut_pythia8_306-1_1_D0_10x100_hiDiv_q2_100_run10_0239_eicrecon_edm4eic.root"
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    # Define visible-yield weights.  These are still e+e- yields, not BR-unfolded Upsilon yields.
    df = add_visible_weight_columns(df)

    # =========================================================================
    # INCLUSIVE PHYSICS SELECTION
    # =========================================================================
    df_signal = df.Filter("reco_mass >= 8.0 && reco_mass <= 12.0") \
                  .Filter("abs(sigma_epz - 36.0) <= 3.6") \
                  .Filter("reco_W >= 0.0")
    #df_signal = df.Filter("reco_W >= 0.0") 
    branches = [
        "reco_mass", "reco_pT", "reco_y", "reco_W", "reco_Q2", "reco_t_proj", "sigma_epz",
        "reco_ePlus_pT", "reco_ePlus_eta", 
        "reco_decay_eMinus_pT", "reco_decay_eMinus_eta",
        "reco_scat_eMinus_E", "reco_scat_eMinus_eta",
        "reco_nTracks_Central", "reco_nTracks_Tagger",
        "weight_visible", "weight_proj_visible"
    ]
    
    print("Pulling inclusive events from RDataFrame into NumPy...")
    data = df_signal.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE PLOT DASHBOARD
    # =========================================================================
    plot_configs = [
        {"var": "reco_mass", "bins": np.linspace(8, 12, 80), "xlabel": r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", "logy": True},
        {"var": "reco_pT", "bins": np.linspace(0, 2.0, 40), "xlabel": r"Reco $\Upsilon$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_y", "bins": np.linspace(-4, 4, 40), "xlabel": r"Reco $\Upsilon$ Rapidity $y$", "logy": False},
        {"var": "reco_W", "bins": np.linspace(20, 150, 65), "xlabel": r"Reco $W_{\gamma p}$ [GeV]", "logy": False},
        
        {"var": "reco_Q2", "bins": np.logspace(-5, 2, 50), "xlabel": r"Reco $Q^2$ [GeV$^2$]", "logy": False, "logx": True},
        {"var": "reco_t_proj", "bins": np.linspace(0, 1.5, 30), "xlabel": r"Projective $|t|$ [GeV$^2$]", "logy": True},
        {"var": "sigma_epz", "bins": np.linspace(25, 45, 60), "xlabel": r"Exclusivity $\Sigma(E-P_z)$ [GeV]", "logy": False},
        
        {"var": "reco_ePlus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^+$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_ePlus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^+$ $\eta$", "logy": False},
        {"var": "reco_decay_eMinus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^-$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_decay_eMinus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^-$ $\eta$", "logy": False},
        
        {"var": "reco_scat_eMinus_E", "bins": np.linspace(0, 25, 50), "xlabel": r"Scattered $e^-$ Energy [GeV]", "logy": False},
        {"var": "reco_scat_eMinus_eta", "bins": np.linspace(-10, 4, 70), "xlabel": r"Scattered $e^-$ $\eta$", "logy": True},
        
        {"var": "reco_nTracks_Central", "bins": np.arange(-0.5, 15.5, 1.0), "xlabel": r"Central Tracks (Count)", "logy": True, "discrete": True},
        {"var": "reco_nTracks_Tagger", "bins": np.arange(-0.5, 5.5, 1.0), "xlabel": r"Low-$Q^2$ Tagger Tracks (Count)", "logy": True, "discrete": True}
    ]

    fig, axs = plt.subplots(4, 4, figsize=(24, 20))
    axs = axs.flatten()

    for i, cfg in enumerate(plot_configs):
        ax = axs[i]
        
        var_name = cfg["var"]
        arr = data[var_name]
        
        wgt = data["weight_proj_visible"] if var_name == "reco_t_proj" else data["weight_visible"]
        
        valid_mask = arr > -900.0
        
        plot_arr = arr[valid_mask]
        plot_wgt = wgt[valid_mask]

        if len(plot_arr) > 0:
            counts, errs = make_hist(plot_arr, cfg["bins"], plot_wgt)
            
            hep.histplot(counts, bins=cfg["bins"], yerr=errs, histtype='step', color='black', ax=ax)
            hep.histplot(counts, bins=cfg["bins"], histtype='fill', color='dodgerblue', alpha=0.4, ax=ax)

        ax.set_xlabel(cfg["xlabel"])
        # Changed y-axis label to match ePIC standard (N_evts)
        ax.set_ylabel(r"$N_{evts}$")
        
        # Add the ePIC labels to every plot!
        draw_epic_labels(ax)
        
        if cfg.get("discrete", False):
            tick_positions = cfg["bins"][:-1] + 0.5
            ax.set_xticks(tick_positions)
        
        # Since we added text to the top left, we MUST expand the y-axis limit 
        # heavily so the data doesn't draw over our new text.
        if cfg.get("logy", False):
            ax.set_yscale('log')
            ax.set_ylim(bottom=0.5)
            if np.max(counts) > 0:
                ax.set_ylim(top=np.max(counts) * 100) # Give 2 orders of magnitude space for text
        else:
            if len(plot_arr) > 0 and np.max(counts) > 0:
                ax.set_ylim(0, np.max(counts) * 1.5) # Expand linear space by 50%
                
        if cfg.get("logx", False):
            ax.set_xscale('log')

    # The 16th panel
    axs[15].axis('off')
    axs[15].text(0.5, 0.6, r"$\Upsilon(nS) \rightarrow e^+e^-$", ha='center', va='center', fontsize=24, fontweight='bold')
    axs[15].text(0.5, 0.4, "Exclusive Reconstructed Kinematics\n" + r"$10$ fb$^{-1}$ Expected visible $e^+e^-$ yield", ha='center', va='center', fontsize=18)

    fig.tight_layout()
    outname = f"Upsilon_Inclusive_Dashboard_{STATE_NORM_MODE}_visibleEE.png"
    fig.savefig(outname, dpi=300)
    print(f"Saved {outname}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Y(1S) reconstructed-kinematics overlay dashboard comparing 10x100 and 18x275.

Layout follows the existing dashboard, but each panel now overlays the same generated
Υ(1S) state for two beam-energy configurations.

The tuple is generated as Υ(1S) -> e+e-. The cross sections used here are visible
e+e- cross sections, so no BR correction is applied.
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use(hep.style.ROOT)

TREE_NAME = "UpsilonTree"
EXPECTED_LUMI_INVFB = 10.0

SAMPLES = {
    "low": {
        "file": "~/merged_elecs_test_10t100_10x100.root",
        "label": "10x100 GeV",
        "campaign": "26.04.1 Campaign",
        "electron_energy": 10.0,
        "visible_xsec_fb": 104.793,
        "color": "dodgerblue",
    },
    "high": {
        "file": "~/merged_elecs_test_tuples_fix2.root",
        "label": "18x275 GeV",
        "campaign": "26.03.1 Campaign",
        "electron_energy": 18.0,
        "visible_xsec_fb": 327.584,
        "color": "darkorange",
    },
}

STATE_NORM_MODE = "target_visible_xsec"  # or "raw"
APPLY_SIGMA_EPZ_CUT = True
SIGMA_EPZ_HALF_WIDTH_FRACTION = 0.20

Y1S_MASS = 9.46030
Y1S_TRUTH_WINDOW = 0.20


def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights * weights)
    return counts, np.sqrt(variances)


def truth_1s_expr():
    return f"true_mass >= {Y1S_MASS - Y1S_TRUTH_WINDOW:.12g} && true_mass <= {Y1S_MASS + Y1S_TRUTH_WINDOW:.12g}"


def draw_epic_labels(ax):
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, r"$\Upsilon(1S)\rightarrow e^+e^-$ comparison",
            transform=ax.transAxes, fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.82, r"$10\times100$ GeV, 26.04.1",
            transform=ax.transAxes, fontsize=10, color=SAMPLES["low"]["color"], ha="left", va="top")
    ax.text(0.04, 0.77, r"$18\times275$ GeV, 26.03.1",
            transform=ax.transAxes, fontsize=10, color=SAMPLES["high"]["color"], ha="left", va="top")
    ax.text(0.04, 0.72, r"$\mathcal{L}_{proj}=10$ fb$^{-1}$, visible $e^+e^-$ yields",
            transform=ax.transAxes, fontsize=10, ha="left", va="top")


def add_visible_weight_columns(df, sample_key):
    cfg = SAMPLES[sample_key]
    scale = 1.0

    if STATE_NORM_MODE == "target_visible_xsec":
        sumw = df.Filter(truth_1s_expr()).Sum("weight").GetValue()
        target_sumw = cfg["visible_xsec_fb"] * EXPECTED_LUMI_INVFB
        scale = target_sumw / sumw if sumw > 0 else 1.0
        raw_xsec = sumw / EXPECTED_LUMI_INVFB if EXPECTED_LUMI_INVFB > 0 else 0.0
        print(f"{cfg['label']} {cfg['campaign']}: raw visible 1S xsec = {raw_xsec:.3f} fb, "
              f"target = {cfg['visible_xsec_fb']:.3f} fb, scale = {scale:.6g}")
    elif STATE_NORM_MODE != "raw":
        raise ValueError(f"Unknown STATE_NORM_MODE: {STATE_NORM_MODE}")

    return (
        df.Define(f"state_norm_{sample_key}", f"{scale}")
          .Define("weight_visible", f"weight * state_norm_{sample_key}")
          .Define("weight_proj_visible", f"weight_proj * state_norm_{sample_key}")
    )


def base_selection(sample_key):
    cfg = SAMPLES[sample_key]
    cuts = [
        "reco_mass >= 8.0 && reco_mass <= 12.0",
        "reco_W >= 0.0",
        f"({truth_1s_expr()})",
    ]

    if APPLY_SIGMA_EPZ_CUT:
        center = 2.0 * cfg["electron_energy"]
        half_width = SIGMA_EPZ_HALF_WIDTH_FRACTION * cfg["electron_energy"]
        cuts.append(f"abs(sigma_epz - {center:.12g}) <= {half_width:.12g}")

    return " && ".join(cuts)


def load_sample_arrays(sample_key, branches):
    cfg = SAMPLES[sample_key]
    try:
        df = ROOT.RDataFrame(TREE_NAME, cfg["file"])
    except Exception as e:
        raise RuntimeError(f"Error opening {cfg['file']}: {e}")

    df = add_visible_weight_columns(df, sample_key)
    df_sel = df.Filter(base_selection(sample_key))
    return df_sel.AsNumpy(branches)


def main():
    print("\n=== Υ(1S) two-energy overlay dashboard ===")
    print(f"STATE_NORM_MODE = {STATE_NORM_MODE}")
    print(f"APPLY_SIGMA_EPZ_CUT = {APPLY_SIGMA_EPZ_CUT}")
    print("No branching-ratio correction is applied.\n")

    branches = [
        "reco_mass", "reco_pT", "reco_y", "reco_W", "reco_Q2", "reco_t_proj", "sigma_epz",
        "reco_ePlus_pT", "reco_ePlus_eta",
        "reco_decay_eMinus_pT", "reco_decay_eMinus_eta",
        "reco_scat_eMinus_E", "reco_scat_eMinus_eta",
        "reco_nTracks_Central", "reco_nTracks_Tagger",
        "weight_visible", "weight_proj_visible",
    ]

    sample_data = {key: load_sample_arrays(key, branches) for key in ["low", "high"]}

    plot_configs = [
        {"var": "reco_mass", "bins": np.linspace(8, 12, 80), "xlabel": r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", "logy": True},
        {"var": "reco_pT", "bins": np.linspace(0, 2.0, 40), "xlabel": r"Reco $\Upsilon(1S)$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_y", "bins": np.linspace(-4, 4, 40), "xlabel": r"Reco $\Upsilon(1S)$ rapidity $y$", "logy": False},
        {"var": "reco_W", "bins": np.linspace(20, 150, 65), "xlabel": r"Reco $W_{\gamma p}$ [GeV]", "logy": False},
        {"var": "reco_Q2", "bins": np.logspace(-5, 2, 50), "xlabel": r"Reco $Q^2$ [GeV$^2$]", "logy": False, "logx": True},
        {"var": "reco_t_proj", "bins": np.linspace(0, 1.5, 30), "xlabel": r"Projective $|t|$ [GeV$^2$]", "logy": True},
        {"var": "sigma_epz", "bins": np.linspace(5, 45, 60), "xlabel": r"Exclusivity $\Sigma(E-P_z)$ [GeV]", "logy": False},
        {"var": "reco_ePlus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^+$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_ePlus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^+$ $\eta$", "logy": False},
        {"var": "reco_decay_eMinus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^-$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_decay_eMinus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^-$ $\eta$", "logy": False},
        {"var": "reco_scat_eMinus_E", "bins": np.linspace(0, 25, 50), "xlabel": r"Scattered $e^-$ energy [GeV]", "logy": False},
        {"var": "reco_scat_eMinus_eta", "bins": np.linspace(-10, 4, 70), "xlabel": r"Scattered $e^-$ $\eta$", "logy": True},
        {"var": "reco_nTracks_Central", "bins": np.arange(-0.5, 15.5, 1.0), "xlabel": r"Central tracks", "logy": True, "discrete": True},
        {"var": "reco_nTracks_Tagger", "bins": np.arange(-0.5, 5.5, 1.0), "xlabel": r"Low-$Q^2$ tagger tracks", "logy": True, "discrete": True},
    ]

    fig, axs = plt.subplots(4, 4, figsize=(24, 20))
    axs = axs.flatten()

    for i, cfg in enumerate(plot_configs):
        ax = axs[i]
        var_name = cfg["var"]
        max_count_for_axis = 0.0

        for sample_key in ["low", "high"]:
            cfg_sample = SAMPLES[sample_key]
            data = sample_data[sample_key]

            arr = data[var_name]
            wgt = data["weight_proj_visible"] if var_name == "reco_t_proj" else data["weight_visible"]

            valid_mask = np.isfinite(arr) & (arr > -900.0)
            if cfg.get("logx", False):
                valid_mask &= arr > 0.0

            plot_arr = arr[valid_mask]
            plot_wgt = wgt[valid_mask]

            if len(plot_arr) == 0:
                continue

            counts, errs = make_hist(plot_arr, cfg["bins"], plot_wgt)
            max_count_for_axis = max(max_count_for_axis, float(np.max(counts)) if len(counts) else 0.0)

            hep.histplot(
                counts,
                bins=cfg["bins"],
                yerr=errs,
                histtype="step",
                color=cfg_sample["color"],
                label=f"{cfg_sample['label']}, {cfg_sample['campaign']}",
                ax=ax,
                linewidth=2,
            )

        ax.set_xlabel(cfg["xlabel"])
        ax.set_ylabel(r"$N_{evts}$")
        draw_epic_labels(ax)

        if cfg.get("discrete", False):
            tick_positions = cfg["bins"][:-1] + 0.5
            ax.set_xticks(tick_positions)

        if cfg.get("logy", False):
            ax.set_yscale("log")
            ax.set_ylim(bottom=0.5)
            if max_count_for_axis > 0:
                ax.set_ylim(top=max_count_for_axis * 100.0)
        else:
            if max_count_for_axis > 0:
                ax.set_ylim(0, max_count_for_axis * 1.6)

        if cfg.get("logx", False):
            ax.set_xscale("log")

        ax.legend(frameon=False, fontsize=9, loc="upper right")

    axs[15].axis("off")
    axs[15].text(0.5, 0.62, r"$\Upsilon(1S) \rightarrow e^+e^-$", ha="center", va="center", fontsize=24, fontweight="bold")
    axs[15].text(0.5, 0.45, "Exclusive Reconstructed Kinematics\n" + r"$10$ fb$^{-1}$ expected visible $e^+e^-$ yield",
                 ha="center", va="center", fontsize=18)
    axs[15].text(0.5, 0.28, r"Overlay: $10\times100$ vs $18\times275$",
                 ha="center", va="center", fontsize=16)

    fig.tight_layout()
    outname = f"Upsilon_1S_TwoEnergy_Overlay_Dashboard_{STATE_NORM_MODE}_visibleEE.png"
    fig.savefig(outname, dpi=300)
    print(f"Saved {outname}")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Side-by-side reconstructed |t| method dashboard for Υ(1S), comparing 10x100 and 18x275.

Style follows the existing two-energy Υ(1S) dashboard:
  - left column: 10x100, campaign 26.04.1
  - right column: 18x275, campaign 26.03.1
  - one row per reconstructed |t| estimator

The samples are Υ(1S) -> e+e-, and the cross sections used here are visible e+e-
cross sections. No BR correction is applied.

The exclusivity cut is applied separately for each energy:
  10x100:  |Sigma(E-pz) - 20| <= 2
  18x275:  |Sigma(E-pz) - 36| <= 3.6
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use(hep.style.ROOT)

TREE_NAME = "UpsilonTree"
EXPECTED_LUMI_INVFB = 10.0

SAMPLES = {
    "low": {
        "file": "~/merged_elecs_test_10t100_10x100.root",
        "label": "10x100 GeV",
        "campaign": "26.04.1 Campaign",
        "electron_energy": 10.0,
        "visible_xsec_fb": 104.793,
        "color": "dodgerblue",
        "w_min": 20.0,
        "w_max": 26.5,
    },
    "high": {
        "file": "~/merged_elecs_test_tuples_fix2.root",
        "label": "18x275 GeV",
        "campaign": "26.03.1 Campaign",
        "electron_energy": 18.0,
        "visible_xsec_fb": 327.584,
        "color": "darkorange",
        "w_min": 37.0,
        "w_max": 114.0,
    },
}

STATE_NORM_MODE = "target_visible_xsec"  # or "raw"
APPLY_SIGMA_EPZ_CUT = True
SIGMA_EPZ_HALF_WIDTH_FRACTION = 0.20

Y1S_MASS = 9.46030
Y1S_TRUTH_WINDOW = 0.20

# Histogram binning for reconstructed |t| comparison.
T_BINS = np.linspace(0.0, 5.0, 51)

METHODS = [
    {
        "key": "eX",
        "branch": "reco_t_eX",
        "weight": "weight_visible",
        "label": r"$t_{eX}$",
        "xlabel": r"Reco $|t_{eX}|$ [GeV$^2$]",
    },
    {
        "key": "BABE",
        "branch": "reco_t_BABE",
        "weight": "weight_visible",
        "label": r"$t_{\rm BABE}$",
        "xlabel": r"Reco $|t_{\rm BABE}|$ [GeV$^2$]",
    },
    {
        "key": "eXBABE",
        "branch": "reco_t_eXBABE",
        "weight": "weight_visible",
        "label": r"$t_{eX{\rm BABE}}$",
        "xlabel": r"Reco $|t_{eX{\rm BABE}}|$ [GeV$^2$]",
    },
    {
        "key": "proj",
        "branch": "reco_t_proj",
        "weight": "weight_proj_visible",
        "label": r"$t_{\rm proj}$",
        "xlabel": r"Projective $|t|$ [GeV$^2$]",
    },
    {
        "key": "pT2",
        "branch": "reco_pT",
        "weight": "weight_visible",
        "label": r"$p_{T,\Upsilon}^{2}$",
        "xlabel": r"Reco $p_{T,\Upsilon}^{2}$ [GeV$^2$]",
    },
]


def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights * weights)
    return counts, np.sqrt(variances)


def truth_1s_expr():
    return f"true_mass >= {Y1S_MASS - Y1S_TRUTH_WINDOW:.12g} && true_mass <= {Y1S_MASS + Y1S_TRUTH_WINDOW:.12g}"


def sigma_epz_cut_expr(sample_key):
    cfg = SAMPLES[sample_key]
    center = 2.0 * cfg["electron_energy"]
    half_width = SIGMA_EPZ_HALF_WIDTH_FRACTION * cfg["electron_energy"]
    return f"abs(sigma_epz - {center:.12g}) <= {half_width:.12g}", center, half_width


def draw_epic_labels(ax, sample_key):
    cfg = SAMPLES[sample_key]
    _, center, half_width = sigma_epz_cut_expr(sample_key)

    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes,
            fontsize=11, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, cfg["campaign"], transform=ax.transAxes,
            fontsize=10, ha="left", va="top")
    ax.text(0.04, 0.82, rf"e+p $\Upsilon(1S)\to e^+e^-$, {cfg['label']}",
            transform=ax.transAxes, fontsize=10, ha="left", va="top")
    ax.text(0.04, 0.76, rf"$W_{{\gamma p}}\in[{cfg['w_min']:g},{cfg['w_max']:g}]$ GeV",
            transform=ax.transAxes, fontsize=10, ha="left", va="top")
    ax.text(0.04, 0.70, rf"$|\Sigma(E-p_z)-{center:g}|<{half_width:g}$ GeV",
            transform=ax.transAxes, fontsize=10, ha="left", va="top")


def add_visible_weight_columns(df, sample_key):
    cfg = SAMPLES[sample_key]
    scale = 1.0

    if STATE_NORM_MODE == "target_visible_xsec":
        sumw = df.Filter(truth_1s_expr()).Sum("weight").GetValue()
        target_sumw = cfg["visible_xsec_fb"] * EXPECTED_LUMI_INVFB
        scale = target_sumw / sumw if sumw > 0 else 1.0
        raw_xsec = sumw / EXPECTED_LUMI_INVFB if EXPECTED_LUMI_INVFB > 0 else 0.0
        print(f"{cfg['label']} {cfg['campaign']}: raw visible 1S xsec = {raw_xsec:.3f} fb, "
              f"target = {cfg['visible_xsec_fb']:.3f} fb, scale = {scale:.6g}")
    elif STATE_NORM_MODE != "raw":
        raise ValueError(f"Unknown STATE_NORM_MODE: {STATE_NORM_MODE}")

    return (
        df.Define(f"state_norm_{sample_key}", f"{scale}")
          .Define("weight_visible", f"weight * state_norm_{sample_key}")
          .Define("weight_proj_visible", f"weight_proj * state_norm_{sample_key}")
          .Define("reco_pT2", "reco_pT * reco_pT")
    )


def base_selection(sample_key):
    cfg = SAMPLES[sample_key]
    cuts = [
        "reco_mass >= 8.0 && reco_mass <= 12.0",
        f"reco_W >= {cfg['w_min']:.12g} && reco_W <= {cfg['w_max']:.12g}",
        f"({truth_1s_expr()})",
    ]

    if APPLY_SIGMA_EPZ_CUT:
        cut, _, _ = sigma_epz_cut_expr(sample_key)
        cuts.append(cut)

    return " && ".join(cuts)


def load_sample_arrays(sample_key, branches):
    cfg = SAMPLES[sample_key]
    try:
        df = ROOT.RDataFrame(TREE_NAME, cfg["file"])
    except Exception as e:
        raise RuntimeError(f"Error opening {cfg['file']}: {e}")

    df = add_visible_weight_columns(df, sample_key)
    df_sel = df.Filter(base_selection(sample_key))
    return df_sel.AsNumpy(branches)


def method_array(data, method_cfg):
    if method_cfg["key"] == "pT2":
        return data["reco_pT2"]
    return data[method_cfg["branch"]]


def main():
    print("\n=== Υ(1S) side-by-side |t| method dashboard ===")
    print(f"STATE_NORM_MODE = {STATE_NORM_MODE}")
    print(f"APPLY_SIGMA_EPZ_CUT = {APPLY_SIGMA_EPZ_CUT}")
    print("No branching-ratio correction is applied.\n")

    branches = [
        "reco_t_eX", "reco_t_BABE", "reco_t_eXBABE", "reco_t_proj",
        "reco_pT", "reco_pT2",
        "weight_visible", "weight_proj_visible",
    ]

    sample_data = {key: load_sample_arrays(key, branches) for key in ["low", "high"]}

    fig, axs = plt.subplots(len(METHODS), 2, figsize=(15, 3.6 * len(METHODS)), sharex=True)
    if len(METHODS) == 1:
        axs = np.array([axs])

    summary_rows = []

    for i, method_cfg in enumerate(METHODS):
        for j, sample_key in enumerate(["low", "high"]):
            ax = axs[i, j]
            cfg_sample = SAMPLES[sample_key]
            data = sample_data[sample_key]

            arr = method_array(data, method_cfg)
            wgt = data[method_cfg["weight"]]

            valid_mask = np.isfinite(arr) & (arr >= T_BINS[0]) & (arr <= T_BINS[-1])
            # Projected-t should only use the wedge-corrected subset.
            if method_cfg["key"] == "proj":
                valid_mask &= np.isfinite(data["weight_proj_visible"]) & (data["weight_proj_visible"] > 0)

            plot_arr = arr[valid_mask]
            plot_wgt = wgt[valid_mask]

            if len(plot_arr) > 0:
                counts, errs = make_hist(plot_arr, T_BINS, plot_wgt)
                density = counts / np.diff(T_BINS)
                density_err = errs / np.diff(T_BINS)

                hep.histplot(density, bins=T_BINS, yerr=density_err, histtype="step",
                             color=cfg_sample["color"], ax=ax, linewidth=2,
                             label=method_cfg["label"])
                hep.histplot(density, bins=T_BINS, histtype="fill",
                             color=cfg_sample["color"], alpha=0.18, ax=ax)

                max_density = float(np.max(density)) if len(density) else 0.0
            else:
                max_density = 0.0

            ax.set_title(f"{cfg_sample['label']}, {method_cfg['label']}", fontsize=16, pad=8)
            ax.set_xlabel(method_cfg["xlabel"], fontsize=13)
            ax.set_ylabel(r"$dN/d|t|$ [GeV$^{-2}$]", fontsize=13)
            ax.set_yscale("log")
            ax.set_ylim(bottom=0.1)
            if max_density > 0:
                ax.set_ylim(top=max_density * 100.0)
            ax.set_xlim(T_BINS[0], T_BINS[-1])
            draw_epic_labels(ax, sample_key)
            ax.legend(frameon=False, fontsize=11, loc="upper right")

            summary_rows.append({
                "sample": sample_key,
                "label": cfg_sample["label"],
                "campaign": cfg_sample["campaign"],
                "method": method_cfg["key"],
                "sum_weight": float(np.sum(plot_wgt)) if len(plot_wgt) else 0.0,
                "n_entries": int(len(plot_arr)),
                "w_min": cfg_sample["w_min"],
                "w_max": cfg_sample["w_max"],
                "sigma_epz_center": 2.0 * cfg_sample["electron_energy"],
                "sigma_epz_half_width": SIGMA_EPZ_HALF_WIDTH_FRACTION * cfg_sample["electron_energy"],
            })

    fig.suptitle(r"Reconstructed $|t|$ estimators for $\Upsilon(1S)$ in two beam-energy samples",
                 fontsize=20, fontweight="bold", y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.985])

    outname = f"Upsilon_1S_TMethods_TwoEnergy_SideBySide_{STATE_NORM_MODE}_visibleEE.png"
    fig.savefig(outname, dpi=300)
    print(f"Saved {outname}")

    # Write a small summary without pandas.
    import csv
    csvname = outname.replace(".png", "_summary.csv")
    with open(csvname, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Saved {csvname}")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Overlay reconstructed kinematic distributions for Υ(1S), Υ(2S), and Υ(3S).

The colors match the mass-fit script:
  Υ(1S): dodgerblue
  Υ(2S): magenta
  Υ(3S): limegreen

The tuple is generated as Υ(nS) -> e+e-, and the target cross sections below are
visible e+e- cross sections. Therefore this script does NOT apply a branching-ratio
correction. It optionally rescales each truth state so sum(weight)/L matches the target
visible cross section.
"""

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use(hep.style.ROOT)

# -------------------------------------------------------------------------
# Input and weight convention
# -------------------------------------------------------------------------
FILENAME = "~/merged_elecs_test_10t100_10x100.root"
TREE_NAME = "UpsilonTree"

EXPECTED_LUMI_INVFB = 10.0

# Visible e+e- eSTARlight cross sections, fb.
TARGET_VISIBLE_XSEC_FB = {
    "1S": 327.584,
    "2S": 84.780,
    "3S": 68.929,
}

# Options:
#   "raw" keeps tuple weights exactly as stored.
#   "target_visible_xsec" rescales each truth state to the target visible e+e- xsec.
STATE_NORM_MODE = "raw"

# -------------------------------------------------------------------------
# State definitions
# -------------------------------------------------------------------------
STATES = {
    "1S": {
        "mass": 9.46030,
        "truth_window": 0.20,
        "color": "dodgerblue",
        "label": r"$\Upsilon(1S)$",
    },
    "2S": {
        "mass": 10.02326,
        "truth_window": 0.20,
        "color": "magenta",
        "label": r"$\Upsilon(2S)$",
    },
    "3S": {
        "mass": 10.35520,
        "truth_window": 0.20,
        "color": "limegreen",
        "label": r"$\Upsilon(3S)$",
    },
}


def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights * weights)
    return counts, np.sqrt(variances)


def draw_epic_labels(ax):
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes,
            fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$",
            transform=ax.transAxes, fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV",
            transform=ax.transAxes, fontsize=11, ha="left", va="top")


def truth_state_expr(mass, window):
    return f"true_mass >= {mass - window:.12g} && true_mass <= {mass + window:.12g}"


def add_state_norm_columns(df):
    """Define one state-normalized visible weight column per Υ state.

    This does not apply a branching-ratio correction.
    """

    scale = {}

    if STATE_NORM_MODE == "target_visible_xsec":
        print("\n--- Visible state-normalization factors ---")
        for state, cfg in STATES.items():
            expr = truth_state_expr(cfg["mass"], cfg["truth_window"])
            sumw = df.Filter(expr).Sum("weight").GetValue()
            target_sumw = TARGET_VISIBLE_XSEC_FB[state] * EXPECTED_LUMI_INVFB
            scale[state] = target_sumw / sumw if sumw > 0 else 1.0
            raw_xsec = sumw / EXPECTED_LUMI_INVFB if EXPECTED_LUMI_INVFB > 0 else 0.0
            print(f"{state}: raw visible xsec = {raw_xsec:.3f} fb, "
                  f"target = {TARGET_VISIBLE_XSEC_FB[state]:.3f} fb, "
                  f"scale = {scale[state]:.6g}")
        print("No BR correction is applied for reconstructed e+e- yields.")
        print("-------------------------------------------\n")
    elif STATE_NORM_MODE == "raw":
        for state in STATES:
            scale[state] = 1.0
    else:
        raise ValueError(f"Unknown STATE_NORM_MODE: {STATE_NORM_MODE}")

    state_norm_expr = (
        f"(fabs(true_mass - {STATES['1S']['mass']}) < {STATES['1S']['truth_window']}) ? ({scale['1S']}) : "
        f"(fabs(true_mass - {STATES['2S']['mass']}) < {STATES['2S']['truth_window']}) ? ({scale['2S']}) : "
        f"(fabs(true_mass - {STATES['3S']['mass']}) < {STATES['3S']['truth_window']}) ? ({scale['3S']}) : "
        "1.0"
    )

    return (
        df.Define("state_norm", state_norm_expr)
          .Define("weight_visible", "weight * state_norm")
          .Define("weight_proj_visible", "weight_proj * state_norm")
    )


def main():
    try:
        df = ROOT.RDataFrame(TREE_NAME, FILENAME)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    df = add_state_norm_columns(df)

    # Common reconstructed selection.
    base_selection = (
        "reco_mass >= 8.0 && reco_mass <= 12.0 && "
        "reco_W >= 0.0"
    )

    branches = [
        "reco_mass", "reco_pT", "reco_y", "reco_W", "reco_Q2", "reco_t_proj", "sigma_epz",
        "reco_ePlus_pT", "reco_ePlus_eta",
        "reco_decay_eMinus_pT", "reco_decay_eMinus_eta",
        "reco_scat_eMinus_E", "reco_scat_eMinus_eta",
        "reco_nTracks_Central", "reco_nTracks_Tagger",
        "weight_visible", "weight_proj_visible",
    ]

    # Plot config dashboard.
    plot_configs = [
        {"var": "reco_mass", "bins": np.linspace(8, 12, 80), "xlabel": r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", "logy": True},
        {"var": "reco_pT", "bins": np.linspace(0, 2.0, 40), "xlabel": r"Reco $\Upsilon$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_y", "bins": np.linspace(-4, 4, 40), "xlabel": r"Reco $\Upsilon$ Rapidity $y$", "logy": False},
        {"var": "reco_W", "bins": np.linspace(20, 150, 65), "xlabel": r"Reco $W_{\gamma p}$ [GeV]", "logy": False},

        {"var": "reco_Q2", "bins": np.logspace(-5, 2, 50), "xlabel": r"Reco $Q^2$ [GeV$^2$]", "logy": False, "logx": True},
        {"var": "reco_t_proj", "bins": np.linspace(0, 1.5, 30), "xlabel": r"Projective $|t|$ [GeV$^2$]", "logy": True},
        {"var": "sigma_epz", "bins": np.linspace(5, 45, 60), "xlabel": r"Exclusivity $\Sigma(E-P_z)$ [GeV]", "logy": False},

        {"var": "reco_ePlus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^+$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_ePlus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^+$ $\eta$", "logy": False},
        {"var": "reco_decay_eMinus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^-$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_decay_eMinus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^-$ $\eta$", "logy": False},

        {"var": "reco_scat_eMinus_E", "bins": np.linspace(0, 25, 50), "xlabel": r"Scattered $e^-$ Energy [GeV]", "logy": False},
        {"var": "reco_scat_eMinus_eta", "bins": np.linspace(-10, 4, 70), "xlabel": r"Scattered $e^-$ $\eta$", "logy": True},

        {"var": "reco_nTracks_Central", "bins": np.arange(-0.5, 15.5, 1.0), "xlabel": r"Central Tracks (Count)", "logy": True, "discrete": True},
        {"var": "reco_nTracks_Tagger", "bins": np.arange(-0.5, 5.5, 1.0), "xlabel": r"Low-$Q^2$ Tagger Tracks (Count)", "logy": True, "discrete": True},
    ]

    fig, axs = plt.subplots(4, 4, figsize=(24, 20))
    axs = axs.flatten()

    for i, cfg in enumerate(plot_configs):
        ax = axs[i]
        var_name = cfg["var"]

        max_count_for_axis = 0.0

        for state, st in STATES.items():
            state_cut = truth_state_expr(st["mass"], st["truth_window"])
            full_cut = f"({base_selection}) && ({state_cut})"
            df_state = df.Filter(full_cut)

            data = df_state.AsNumpy(branches)
            arr = data[var_name]
            wgt = data["weight_proj_visible"] if var_name == "reco_t_proj" else data["weight_visible"]

            valid_mask = arr > -900.0
            plot_arr = arr[valid_mask]
            plot_wgt = wgt[valid_mask]

            if len(plot_arr) == 0:
                continue

            counts, errs = make_hist(plot_arr, cfg["bins"], plot_wgt)
            max_count_for_axis = max(max_count_for_axis, float(np.max(counts)) if len(counts) else 0.0)

            hep.histplot(
                counts,
                bins=cfg["bins"],
                yerr=errs,
                histtype="step",
                color=st["color"],
                label=st["label"],
                ax=ax,
                linewidth=2,
            )

        ax.set_xlabel(cfg["xlabel"])
        ax.set_ylabel(r"$N_{evts}$")
        draw_epic_labels(ax)

        if cfg.get("discrete", False):
            tick_positions = cfg["bins"][:-1] + 0.5
            ax.set_xticks(tick_positions)

        if cfg.get("logy", False):
            ax.set_yscale("log")
            ax.set_ylim(bottom=0.5)
            if max_count_for_axis > 0:
                ax.set_ylim(top=max_count_for_axis * 100.0)
        else:
            if max_count_for_axis > 0:
                ax.set_ylim(0, max_count_for_axis * 1.6)

        if cfg.get("logx", False):
            ax.set_xscale("log")

        ax.legend(frameon=False, fontsize=9, loc="upper right")

    axs[15].axis("off")
    axs[15].text(0.5, 0.62, r"$\Upsilon(nS) \rightarrow e^+e^-$", ha="center", va="center", fontsize=24, fontweight="bold")
    axs[15].text(0.5, 0.45, "Exclusive Reconstructed Kinematics\n"
                              + r"$10$ fb$^{-1}$ Expected visible $e^+e^-$ yield",
                 ha="center", va="center", fontsize=18)
    axs[15].text(0.5, 0.28, r"Overlaid by generated $\Upsilon(nS)$ state",
                 ha="center", va="center", fontsize=16)

    fig.tight_layout()
    outname = f"Upsilon_ThreeState_Overlay_Dashboard_{STATE_NORM_MODE}_visibleEE.png"
    fig.savefig(outname, dpi=300)
    print(f"Saved {outname}")


if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from scipy.special import j0

# Use standard ROOT style, but we will draw our own text blocks
hep.style.use(hep.style.ROOT)

# Conversion constant: hbar * c in GeV * fm
HBARC = 0.1973269804

def make_hist(data, bins, weights):
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights**2)
    return counts, np.sqrt(variances)

def transform_t_to_b(t_centers, dsigma_dt, b_max=10.0, n_b_bins=10):
    """
    Transforms a |t| distribution into an impact parameter F(b_T) distribution.
   
    """
    dt = np.gradient(t_centers)
    amplitude = np.sqrt(np.maximum(dsigma_dt, 0.0))
    b_values = np.linspace(0, b_max, n_b_bins)
    F_b = np.zeros_like(b_values, dtype=float)
    
    for i, b in enumerate(b_values):
        arg = (b * np.sqrt(t_centers)) / HBARC
        integrand = j0(arg) * amplitude
        F_b[i] = (1.0 / (4.0 * np.pi)) * np.sum(integrand * dt)
        
    if np.max(F_b) > 0:
        F_b = F_b / np.max(F_b)
        
    return b_values, F_b

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top')
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 3. Reaction and Energy
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')
    
    # 4. Luminosity and sqrt(s)
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top')

def main():
    filename = "~/merged_elecs_test_tuples.root"
    # filename = "~/AnalysisOut_EpIC1_1_6-1_0_DDVCS_18x275_q2_0_10_edecay_hplus_abconv_run0_eicrecon_edm4eic.root"
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file: {e}")
        return

    # =========================================================================
    # INCLUSIVE PHYSICS SELECTION
    # =========================================================================
    df_signal = df.Filter("reco_mass >= 8.0 && reco_mass <= 12.0") \
                  .Filter("abs(sigma_epz - 36.0) <= 3.6") \
                  .Filter("reco_W >= 0.0")
    branches = [
        "reco_mass", "reco_pT", "reco_y", "reco_W", "reco_Q2", "reco_t_proj", "sigma_epz",
        "reco_ePlus_pT", "reco_ePlus_eta", 
        "reco_decay_eMinus_pT", "reco_decay_eMinus_eta",
        "reco_scat_eMinus_E", "reco_scat_eMinus_eta",
        "reco_nTracks_Central", "reco_nTracks_Tagger",
        "weight", "weight_proj"
    ]
    
    print("Pulling inclusive events from RDataFrame into NumPy...")
    data = df_signal.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE PLOT DASHBOARD
    # =========================================================================
    plot_configs = [
        {"var": "reco_mass", "bins": np.linspace(8, 12, 80), "xlabel": r"Reco $M_{e^+e^-}$ [GeV/$c^2$]", "logy": False},
        {"var": "reco_pT", "bins": np.linspace(0, 2.0, 40), "xlabel": r"Reco $\Upsilon$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_y", "bins": np.linspace(-4, 4, 40), "xlabel": r"Reco $\Upsilon$ Rapidity $y$", "logy": False},
        {"var": "reco_W", "bins": np.linspace(20, 150, 65), "xlabel": r"Reco $W_{\gamma p}$ [GeV]", "logy": False},
        
        {"var": "reco_Q2", "bins": np.logspace(-5, 2, 50), "xlabel": r"Reco $Q^2$ [GeV$^2$]", "logy": False, "logx": True},
        {"var": "reco_t_proj", "bins": np.linspace(0, 1.5, 60), "xlabel": r"Projective $|t|$ [GeV$^2$]", "logy": True},
        {"var": "sigma_epz", "bins": np.linspace(25, 45, 60), "xlabel": r"Exclusivity $\Sigma(E-P_z)$ [GeV]", "logy": False},
        
        {"var": "reco_ePlus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^+$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_ePlus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^+$ $\eta$", "logy": False},
        {"var": "reco_decay_eMinus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"Decay $e^-$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "reco_decay_eMinus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"Decay $e^-$ $\eta$", "logy": False},
        
        {"var": "reco_scat_eMinus_E", "bins": np.linspace(0, 25, 50), "xlabel": r"Scattered $e^-$ Energy [GeV]", "logy": False},
        {"var": "reco_scat_eMinus_eta", "bins": np.linspace(-10, 4, 70), "xlabel": r"Scattered $e^-$ $\eta$", "logy": True},
        
        {"var": "reco_nTracks_Central", "bins": np.arange(-0.5, 15.5, 1.0), "xlabel": r"Central Tracks (Count)", "logy": True, "discrete": True},
        {"var": "reco_nTracks_Tagger", "bins": np.arange(-0.5, 5.5, 1.0), "xlabel": r"Low-$Q^2$ Tagger Tracks (Count)", "logy": True, "discrete": True},
        
        # New configuration for the Fourier Transform
        {"var": "fourier_bT", "bins": None, "xlabel": r"Impact Parameter $b_T$ [fm]", "logy": True, "is_transform": True}
    ]

    # Adjusted to 4x5 to fit 16 plots + Title Panel + 3 empty spaces
    fig, axs = plt.subplots(4, 5, figsize=(30, 20))
    axs = axs.flatten()

    for i, cfg in enumerate(plot_configs):
        ax = axs[i]
        var_name = cfg["var"]
        
        # Handle the derived Fourier transform plot
        if cfg.get("is_transform", False):
            # We need the data from reco_t_proj to perform the transform
            arr = data["reco_t_proj"]
            wgt = data["weight_proj"]
            valid_mask = arr > -900.0
            plot_arr = arr[valid_mask]
            plot_wgt = wgt[valid_mask]
            
            if len(plot_arr) > 0:
                # 1. Create the |t| histogram
                t_bins = np.linspace(0, 1.5, 30)
                t_counts, _ = make_hist(plot_arr, t_bins, plot_wgt)
                t_centers = (t_bins[:-1] + t_bins[1:]) / 2
                
                # 2. Perform the Fourier-Bessel Transform
                b_vals, F_b_vals = transform_t_to_b(t_centers, t_counts, b_max=10.0, n_b_bins=100)
                
                # 3. Plot the result
                ax.plot(b_vals, F_b_vals, color='black', linewidth=2)
                ax.fill_between(b_vals, 0, F_b_vals, color='dodgerblue', alpha=0.4)
                ax.plot(b_vals, F_b_vals, color='black', linewidth=2)
                ax.fill_between(b_vals, 0, F_b_vals, color='dodgerblue', alpha=0.4)
                ax.set_ylabel(r"$F(b_T)$ (normalized)")
                # Set a linear scale with some padding
                ax.set_ylim(0, np.max(F_b_vals) * 1.5)
                
        # Handle standard RDataFrame variables
        else:
            arr = data[var_name]
            wgt = data["weight_proj"] if var_name == "reco_t_proj" else data["weight"]
            valid_mask = arr > -900.0
            plot_arr = arr[valid_mask]
            plot_wgt = wgt[valid_mask]

            if len(plot_arr) > 0:
                counts, errs = make_hist(plot_arr, cfg["bins"], plot_wgt)
                
                hep.histplot(counts, bins=cfg["bins"], yerr=errs, histtype='step', color='black', ax=ax)
                hep.histplot(counts, bins=cfg["bins"], histtype='fill', color='dodgerblue', alpha=0.4, ax=ax)

            ax.set_ylabel(r"$N_{evts}$")
            
            if cfg.get("discrete", False):
                tick_positions = cfg["bins"][:-1] + 0.5
                ax.set_xticks(tick_positions)
            
            if cfg.get("logy", False):
                ax.set_yscale('log')
                ax.set_ylim(bottom=0.5)
                if np.max(counts) > 0:
                    ax.set_ylim(top=np.max(counts) * 100) 
            else:
                if len(plot_arr) > 0 and np.max(counts) > 0:
                    ax.set_ylim(0, np.max(counts) * 1.5) 

            if cfg.get("logx", False):
                ax.set_xscale('log')

        # Standard formatting applied to all plots
        ax.set_xlabel(cfg["xlabel"])
        draw_epic_labels(ax)

    # Hide unused subplots (panels 17, 18, 19)
    for j in range(16, 19):
        axs[j].axis('off')

    # The final panel for the title
    axs[19].axis('off')
    axs[19].text(0.5, 0.6, r"$\Upsilon(nS) \rightarrow e^+e^-$", ha='center', va='center', fontsize=24, fontweight='bold')
    axs[19].text(0.5, 0.4, "Exclusive Reconstructed Kinematics\n" + r"$10$ fb$^{-1}$ Expected Yield", ha='center', va='center', fontsize=18)

    fig.tight_layout()
    fig.savefig("Upsilon_Inclusive_Dashboard.png", dpi=300)
    print("Saved Upsilon_Inclusive_Dashboard.png")

if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style
hep.style.use(hep.style.ROOT)

def make_hist(data, bins, weights=None):
    """Creates histogram counts and poisson/weighted errors."""
    if weights is None:
        weights = np.ones_like(data)
    counts, _ = np.histogram(data, bins=bins, weights=weights)
    variances, _ = np.histogram(data, bins=bins, weights=weights**2)
    return counts, np.sqrt(variances)

def draw_epic_labels(ax, add_bg=False):
    """Draws the standard ePIC performance block in the top left corner."""
    bbox_props = dict(boxstyle="square,pad=0.0", facecolor="white", edgecolor="none", alpha=0.75) if add_bg else None

    # 1. ePIC Performance (Bold)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', ha='left', va='top', bbox=bbox_props)
    
    # 2. Campaign
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top', bbox=bbox_props)
    
    # 3. Reaction
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$", transform=ax.transAxes, 
            fontsize=11, ha='left', va='top', bbox=bbox_props)

def plot_2d_contour_overlay(ax, x10, y10, w10, x18, y18, w18, xbins, ybins, xlabel, ylabel, title):
    """Helper to overlay two 2D distributions as contour lines."""
    # Compute 2D histograms
    h10, xedges, yedges = np.histogram2d(x10, y10, bins=[xbins, ybins], weights=w10)
    h18, _, _ = np.histogram2d(x18, y18, bins=[xbins, ybins], weights=w18)
    
    # Get bin centers for meshgrid
    xc = (xedges[:-1] + xedges[1:]) / 2
    yc = (yedges[:-1] + yedges[1:]) / 2
    X, Y = np.meshgrid(xc, yc)
    
    # Draw 10x100 Contours (Orange)
    if np.max(h10) > 0:
        # Create 6 contour levels from 5% of max to max
        levels10 = np.linspace(np.max(h10)*0.05, np.max(h10), 6)
        # Note: h.T is required because contour expects Z as (ny, nx)
        ax.contour(X, Y, h10.T, levels=levels10, colors='darkorange', linewidths=2, alpha=0.9)
        # Fake plot line just for the legend
        ax.plot([], [], color='darkorange', linewidth=2, label='10x100 GeV')
        
    # Draw 18x275 Contours (Blue)
    if np.max(h18) > 0:
        levels18 = np.linspace(np.max(h18)*0.05, np.max(h18), 6)
        ax.contour(X, Y, h18.T, levels=levels18, colors='dodgerblue', linewidths=2, alpha=0.9)
        ax.plot([], [], color='dodgerblue', linewidth=2, label='18x275 GeV')

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=14, pad=10)
    ax.legend(loc="upper right", frameon=False, fontsize=11)
    draw_epic_labels(ax, add_bg=True)

def main():
    # Input files
    file_10x100 = "~/merged_elecs_test_10t100_10x100.root"
    file_18x275 = "~/merged_elecs_test_tuples_fix2.root"
    
    try:
        df_10x100 = ROOT.RDataFrame("UpsilonTree", file_10x100)
        df_18x275 = ROOT.RDataFrame("UpsilonTree", file_18x275)
    except Exception as e:
        print(f"Error opening files: {e}")
        return

    branches = [
        "true_mass", "true_pT", "true_y", "true_W", "true_Q2", "true_t",
        "true_ePlus_pT", "true_ePlus_eta", 
        "true_decay_eMinus_pT", "true_decay_eMinus_eta",
        "true_scat_eMinus_E", "true_scat_eMinus_eta",
        "weight"
    ]
    
    print("Pulling true-level events from RDataFrame into NumPy...")
    data_10x100 = df_10x100.AsNumpy(branches)
    data_18x275 = df_18x275.AsNumpy(branches)

    # =========================================================================
    # DEFINE THE 12 1D PLOTS
    # =========================================================================
    plot_configs = [
        {"var": "true_mass", "bins": np.linspace(8, 12, 80), "xlabel": r"True $M_{e^+e^-}$ [GeV/$c^2$]", "logy": True},
        {"var": "true_pT", "bins": np.linspace(0, 2.0, 40), "xlabel": r"True $\Upsilon$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "true_y", "bins": np.linspace(-4, 4, 40), "xlabel": r"True $\Upsilon$ Rapidity $y$", "logy": False},
        {"var": "true_W", "bins": np.linspace(20, 150, 65), "xlabel": r"True $W_{\gamma p}$ [GeV]", "logy": False},
        
        {"var": "true_Q2", "bins": np.logspace(-5, 2, 50), "xlabel": r"True $Q^2$ [GeV$^2$]", "logy": False, "logx": True},
        {"var": "true_t", "bins": np.linspace(0, 1.5, 30), "xlabel": r"True $|t|$ [GeV$^2$]", "logy": True},
        {"var": "true_scat_eMinus_E", "bins": np.linspace(0, 30, 50), "xlabel": r"True Scattered $e^-$ Energy [GeV]", "logy": False},
        {"var": "true_scat_eMinus_eta", "bins": np.linspace(-10, 4, 70), "xlabel": r"True Scattered $e^-$ $\eta$", "logy": True},
        
        {"var": "true_ePlus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"True Decay $e^+$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "true_ePlus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"True Decay $e^+$ $\eta$", "logy": False},
        {"var": "true_decay_eMinus_pT", "bins": np.linspace(0, 10, 40), "xlabel": r"True Decay $e^-$ $p_T$ [GeV/$c$]", "logy": False},
        {"var": "true_decay_eMinus_eta", "bins": np.linspace(-4, 4, 40), "xlabel": r"True Decay $e^-$ $\eta$", "logy": False},
    ]

    fig, axs = plt.subplots(4, 4, figsize=(24, 20))
    axs = axs.flatten()

    # --- 1. Plot the 12 1D Histograms ---
    for i, cfg in enumerate(plot_configs):
        ax = axs[i]
        var_name = cfg["var"]
        
        arr_10 = data_10x100[var_name]
        wgt_10 = data_10x100["weight"]
        valid_10 = arr_10 > -900.0
        
        arr_18 = data_18x275[var_name]
        wgt_18 = data_18x275["weight"]
        valid_18 = arr_18 > -900.0

        max_y = 0

        # Plot 10x100 (Orange)
        if len(arr_10[valid_10]) > 0:
            c_10, err_10 = make_hist(arr_10[valid_10], cfg["bins"], wgt_10[valid_10])
            hep.histplot(c_10, bins=cfg["bins"], yerr=err_10, histtype='step', 
                         color='darkorange', label='10x100 GeV', ax=ax, lw=2)
            max_y = max(max_y, np.max(c_10))

        # Plot 18x275 (Blue)
        if len(arr_18[valid_18]) > 0:
            c_18, err_18 = make_hist(arr_18[valid_18], cfg["bins"], wgt_18[valid_18])
            hep.histplot(c_18, bins=cfg["bins"], yerr=err_18, histtype='step', 
                         color='dodgerblue', label='18x275 GeV', ax=ax, lw=2)
            max_y = max(max_y, np.max(c_18))

        ax.set_xlabel(cfg["xlabel"])
        ax.set_ylabel(r"$N_{evts}$ (True Level)")
        
        draw_epic_labels(ax)
        
        if cfg.get("logy", False):
            ax.set_yscale('log')
            ax.set_ylim(bottom=0.5)
            if max_y > 0: ax.set_ylim(top=max_y * 500)
        else:
            if max_y > 0: ax.set_ylim(0, max_y * 1.6) 
                
        if cfg.get("logx", False):
            ax.set_xscale('log')
            
        ax.legend(loc="upper right", frameon=False, fontsize=11)

    # --- 2. Overlay 2D Contour Plots ---
    
    # Panel 13: Decay pT Correlation
    v10_corr = (data_10x100["true_decay_eMinus_pT"] > -900) & (data_10x100["true_ePlus_pT"] > -900)
    v18_corr = (data_18x275["true_decay_eMinus_pT"] > -900) & (data_18x275["true_ePlus_pT"] > -900)
    
    plot_2d_contour_overlay(
        ax=axs[12],
        x10=data_10x100["true_decay_eMinus_pT"][v10_corr], y10=data_10x100["true_ePlus_pT"][v10_corr], w10=data_10x100["weight"][v10_corr],
        x18=data_18x275["true_decay_eMinus_pT"][v18_corr], y18=data_18x275["true_ePlus_pT"][v18_corr], w18=data_18x275["weight"][v18_corr],
        xbins=np.linspace(0, 12, 60), ybins=np.linspace(0, 12, 60),
        xlabel=r"True Decay $e^-$ $p_T$ [GeV/$c$]", ylabel=r"True Decay $e^+$ $p_T$ [GeV/$c$]",
        title="Gen Decay Leptons $p_T$ Correlation"
    )

    # Panel 14: Decay eta vs pT (Combined e+ and e- arrays)
    pt_comb_10 = np.concatenate([data_10x100["true_ePlus_pT"], data_10x100["true_decay_eMinus_pT"]])
    eta_comb_10 = np.concatenate([data_10x100["true_ePlus_eta"], data_10x100["true_decay_eMinus_eta"]])
    w_comb_10 = np.concatenate([data_10x100["weight"], data_10x100["weight"]])
    v10_eta = (pt_comb_10 > -900) & (eta_comb_10 > -900)

    pt_comb_18 = np.concatenate([data_18x275["true_ePlus_pT"], data_18x275["true_decay_eMinus_pT"]])
    eta_comb_18 = np.concatenate([data_18x275["true_ePlus_eta"], data_18x275["true_decay_eMinus_eta"]])
    w_comb_18 = np.concatenate([data_18x275["weight"], data_18x275["weight"]])
    v18_eta = (pt_comb_18 > -900) & (eta_comb_18 > -900)

    plot_2d_contour_overlay(
        ax=axs[13],
        x10=pt_comb_10[v10_eta], y10=eta_comb_10[v10_eta], w10=w_comb_10[v10_eta],
        x18=pt_comb_18[v18_eta], y18=eta_comb_18[v18_eta], w18=w_comb_18[v18_eta],
        xbins=np.linspace(0, 10, 50), ybins=np.linspace(-5, 5, 50),
        xlabel=r"True Decay Lepton $p_T$ [GeV/$c$]", ylabel=r"True Decay Lepton $\eta$",
        title=r"Gen Decay Leptons $\eta$ vs $p_T$"
    )

    # Panel 15: |t| vs W Correlation
    # We use the full W axis (20 to 150) so both energies map to the same grid!
    # The 10x100 will naturally only draw contours up to ~63 GeV.
    v10_tw = (data_10x100["true_W"] > -900) & (data_10x100["true_t"] > -900)
    v18_tw = (data_18x275["true_W"] > -900) & (data_18x275["true_t"] > -900)

    plot_2d_contour_overlay(
        ax=axs[14],
        x10=data_10x100["true_W"][v10_tw], y10=data_10x100["true_t"][v10_tw], w10=data_10x100["weight"][v10_tw],
        x18=data_18x275["true_W"][v18_tw], y18=data_18x275["true_t"][v18_tw], w18=data_18x275["weight"][v18_tw],
        xbins=np.linspace(20, 150, 65), ybins=np.linspace(0, 1.5, 30),
        xlabel=r"True $W_{\gamma p}$ [GeV]", ylabel=r"True $|t|$ [GeV$^2$]",
        title=r"Gen $|t|$ vs $W_{\gamma p}$ Correlation"
    )

    # --- 3. Text Block (Panel 16) ---
    axs[15].axis('off')
    axs[15].text(0.5, 0.6, r"$\Upsilon(nS) \rightarrow e^+e^-$", ha='center', va='center', fontsize=24, fontweight='bold')
    axs[15].text(0.5, 0.4, "True Level Kinematics\nBeam Energy 1D & 2D Overlays", ha='center', va='center', fontsize=18)

    fig.tight_layout()
    fig.savefig("Upsilon_TrueLevel_Overlay_Dashboard.png", dpi=300)
    print("Saved Upsilon_TrueLevel_Overlay_Dashboard.png")

if __name__ == "__main__":
    main()

In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style
hep.style.use(hep.style.ROOT)

def get_th1d_data(file_obj, hist_name):
    """Extracts bin contents, edges, and errors from a ROOT TH1D."""
    h = file_obj.Get(hist_name)
    if not h or not isinstance(h, ROOT.TH1):
        print(f"Warning: Could not retrieve TH1D '{hist_name}' from {file_obj.GetName()}")
        return None, None, None
        
    n_bins = h.GetNbinsX()
    
    # Extract edges (n_bins + 1)
    edges = np.array([h.GetBinLowEdge(i) for i in range(1, n_bins + 2)])
    # Extract contents and errors (n_bins)
    contents = np.array([h.GetBinContent(i) for i in range(1, n_bins + 1)])
    errors = np.array([h.GetBinError(i) for i in range(1, n_bins + 1)])
    
    return contents, edges, errors

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=14, fontweight='bold', ha='left', va='top')
    
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=13, ha='left', va='top')
    
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$", transform=ax.transAxes, 
            fontsize=13, ha='left', va='top')

def calc_pct_lost(c):
    """Calculates the percentage of events lost relative to the previous bin."""
    pct_lost = np.zeros_like(c, dtype=float)
    for i in range(1, len(c)):
        if c[i-1] > 0:
            pct_lost[i] = ((c[i-1] - c[i]) / c[i-1]) * 100.0
    return pct_lost

def main():
    # Input files
    file_10x100_path = "~/merged_elecs_test_10t100.root"
    file_18x275_path = "~/merged_elecs_test_tuples.root"
    
    f_10 = ROOT.TFile.Open(file_10x100_path, "READ")
    f_18 = ROOT.TFile.Open(file_18x275_path, "READ")
    
    if not f_10 or f_10.IsZombie() or not f_18 or f_18.IsZombie():
        print("Error opening ROOT files.")
        return

    hist_name = "h_CutFlow"
    
    # Create a figure with 2 subplots
    fig, (ax_main, ax_loss) = plt.subplots(
        2, 1, figsize=(12, 10), 
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}, 
        sharex=True
    )
    
    fig.suptitle(r"$\Upsilon(nS) \rightarrow e^+e^-$ | Event Selection Cutflow", 
                 fontsize=22, fontweight='bold', y=0.98)
    
    # Pull the pre-binned data from the TH1Ds
    c_10, bins_10, err_10 = get_th1d_data(f_10, hist_name)
    c_18, bins_18, err_18 = get_th1d_data(f_18, hist_name)

    max_y = 0

    # --- TOP PANEL: Main Cutflow ---
    if c_10 is not None:
        hep.histplot(c_10, bins=bins_10, yerr=err_10, histtype='step', 
                     color='darkorange', label='10x100 GeV', ax=ax_main, lw=2)
        max_y = max(max_y, np.max(c_10))

    if c_18 is not None:
        hep.histplot(c_18, bins=bins_18, yerr=err_18, histtype='step', 
                     color='dodgerblue', label='18x275 GeV', ax=ax_main, lw=2)
        max_y = max(max_y, np.max(c_18))

    ax_main.set_ylabel("Remaining Events", fontsize=16)
    draw_epic_labels(ax_main)
    
    ax_main.set_yscale('log')
    ax_main.set_ylim(bottom=0.5)
    if max_y > 0: 
        ax_main.set_ylim(top=max_y * 200) # Ensure legend/text space
            
    ax_main.legend(loc="upper right", frameon=False, fontsize=14)


    # --- BOTTOM PANEL: Percentage Lost ---
    if c_10 is not None and c_18 is not None:
        pct_lost_10 = calc_pct_lost(c_10)
        pct_lost_18 = calc_pct_lost(c_18)
        
        hep.histplot(pct_lost_10, bins=bins_10, histtype='step', color='darkorange', ax=ax_loss, lw=2)
        hep.histplot(pct_lost_18, bins=bins_18, histtype='step', color='dodgerblue', ax=ax_loss, lw=2)
        
        # Gridlines for readability
        for y_val in [0, 25, 50, 75, 100]:
            ax_loss.axhline(y_val, color='gray', linestyle='--', alpha=0.3, lw=1)

    ax_loss.set_ylabel("% Lost\n(per step)", fontsize=14)
    ax_loss.set_ylim(-5, 110) 

    # --- X-AXIS LABELS ---
    cut_names = [
        "1. Total Processed", 
        "2. Gen Upsilon Found", 
        "3. Gen Leps in Acc.",
        "4. Reco e+ Candidate", 
        "5. Reco e- Decay Cand", 
        "6. Reco Scat. e- Cand",
        "7. Mass Window [8, 12]", 
        r"8. $\Sigma(E-P_z)$ Exclusivity", 
        "9. W Phase Space Cut", 
        "10. Proton Tagged"
    ]
    
    ax_loss.set_xticks(np.arange(1, 11))
    ax_loss.set_xticklabels(cut_names, rotation=45, ha='right', fontsize=12)

    # Clean up layout explicitly to fix the cutoff issue
    # We call tight_layout first to handle internal spacing, then override margins
    fig.tight_layout()
    # explicitly reserve 15% of the figure height at the bottom, and 6% at the top
    fig.subplots_adjust(bottom=0.15, top=0.94) 
    
    fig.savefig("Upsilon_EventSelection_Cutflow_with_Loss.png", dpi=300)
    print("Saved Upsilon_EventSelection_Cutflow_with_Loss.png")

    f_10.Close()
    f_18.Close()

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import math
import warnings

import numpy as np
import pandas as pd
import ROOT

ROOT.gROOT.SetBatch(True)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    import mplhep as hep
    HAS_MPLHEP = True
except Exception:
    HAS_MPLHEP = False

try:
    from scipy.special import j0 as scipy_j0
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

HBARC_GEV_FM = 0.1973269804

METHODS = {
    "true": {
        "label": "generator truth |t|",
        "branch": "true_t",
        "weight": "weight",
        "extra_cut": "true_t > 0",
    },
    "eX": {
        "label": "lepton + Upsilon, t_eX",
        "branch": "reco_t_eX",
        "weight": "weight",
        "extra_cut": "reco_t_eX > 0",
    },
    "BABE": {
        "label": "proton-only, t_BABE",
        "branch": "reco_t_BABE",
        "weight": "weight",
        "extra_cut": "reco_t_BABE > 0",
    },
    "eXBABE": {
        "label": "hybrid, t_eXBABE",
        "branch": "reco_t_eXBABE",
        "weight": "weight",
        "extra_cut": "reco_t_eXBABE > 0",
    },
    "proj": {
        "label": "projected-t wedge",
        "branch": "reco_t_proj",
        "weight": "weight_proj",
        "extra_cut": "reco_t_proj > 0 && weight_proj > 0",
    },
    "pT2": {
        "label": "Upsilon pT^2 proxy",
        "branch": "reco_pT",
        "weight": "weight",
        "extra_cut": "reco_pT > 0",
    },
}

def parse_float_list(s):
    return [float(x) for x in s.split(",") if x.strip()]

def bessel_j0(x):
    if HAS_SCIPY:
        return scipy_j0(x)
    vec = np.vectorize(lambda z: ROOT.TMath.BesselJ0(float(z)))
    return vec(x)

def weighted_quantile(values, quantiles, sample_weight=None):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    if sample_weight is None:
        return np.quantile(values, quantiles)
    sample_weight = np.asarray(sample_weight, dtype=float)
    mask = np.isfinite(values) & np.isfinite(sample_weight) & (sample_weight > 0)
    values = values[mask]
    sample_weight = sample_weight[mask]
    if len(values) == 0:
        return np.full_like(quantiles, np.nan, dtype=float)
    order = np.argsort(values)
    values = values[order]
    sample_weight = sample_weight[order]
    cdf = np.cumsum(sample_weight)
    cdf /= cdf[-1]
    return np.interp(quantiles, cdf, values)

def build_calibrated_reco_t(df, mode="none", n_knots=12, seed=12345, train_fraction=1.0):
    reco = np.asarray(df["t_raw"], dtype=float)
    if mode == "none":
        curve = pd.DataFrame({"calibration": ["none"], "x_reco": [np.nan], "y_true": [np.nan]})
        return reco, curve
    if "true_t" not in df.columns:
        raise RuntimeError("Calibration requested, but true_t is not available.")
    true = np.asarray(df["true_t"], dtype=float)
    w = np.asarray(df["weight"], dtype=float)
    mask = np.isfinite(reco) & np.isfinite(true) & np.isfinite(w) & (reco > 0) & (true > 0) & (w > 0)
    if train_fraction < 1.0:
        rng = np.random.default_rng(seed)
        mask = mask & (rng.random(len(df)) < train_fraction)
    if np.sum(mask) < 10:
        warnings.warn("Too few events for calibration. Returning uncalibrated reco_t.")
        curve = pd.DataFrame({"calibration": [mode], "x_reco": [np.nan], "y_true": [np.nan]})
        return reco, curve
    r = reco[mask]
    t = true[mask]
    ww = w[mask]
    if mode == "scale":
        scale = float(weighted_quantile(t / r, [0.5], ww)[0])
        curve = pd.DataFrame({"calibration": ["scale"], "x_reco": [1.0], "y_true": [scale]})
        return reco * scale, curve
    if mode == "quantile":
        probs = np.linspace(0.02, 0.98, n_knots)
        xk = weighted_quantile(r, probs, ww)
        yk = weighted_quantile(t, probs, ww)
        keep = np.isfinite(xk) & np.isfinite(yk)
        xk = xk[keep]
        yk = yk[keep]
        probs = probs[keep]
        xk_unique, idx = np.unique(xk, return_index=True)
        xk = xk_unique
        yk = yk[idx]
        probs = probs[idx]
        if len(xk) < 2:
            warnings.warn("Too few unique quantile knots. Returning uncalibrated reco_t.")
            curve = pd.DataFrame({"calibration": ["quantile"], "prob": [np.nan], "x_reco": [np.nan], "y_true": [np.nan]})
            return reco, curve
        reco_cal = np.interp(reco, xk, yk, left=yk[0], right=yk[-1])
        curve = pd.DataFrame({"calibration": "quantile", "prob": probs, "x_reco": xk, "y_true": yk})
        return reco_cal, curve
    raise ValueError(f"Unknown calibration mode: {mode}")

def load_tuple_dataframe(path, tree_name, method, args):
    path = os.path.expanduser(path)
    cfg = METHODS[method]
    t_branch = cfg["branch"]
    needed = sorted(set([t_branch, cfg["weight"], "true_t", "reco_mass", "sigma_epz", "reco_W", "weight"]))
    cuts = [cfg["extra_cut"], f"{cfg['weight']} > 0", f"reco_mass >= {args.mass_min:.12g}", f"reco_mass <= {args.mass_max:.12g}", f"reco_W >= {args.w_min:.12g}"]
    if args.apply_sigma_epz:
        cuts.append(f"abs(sigma_epz - {2.0 * args.electron_energy:.12g}) <= {args.sigma_epz_half_width:.12g}")
    if method == "proj" and args.require_proton_tag_for_proj:
        cuts.append("reco_t_eXBABE > 0")
    cut = " && ".join([c for c in cuts if c])
    rdf = ROOT.RDataFrame(tree_name, path).Filter(cut)
    arrs = rdf.AsNumpy(columns=needed)
    if len(arrs) == 0 or len(next(iter(arrs.values()))) == 0:
        raise RuntimeError("No events passed the selection.")
    t_raw = np.asarray(arrs[t_branch], dtype=float)
    if method == "pT2":
        t_raw = t_raw * t_raw
    if method == "proj" and not args.use_corrected_proj_weight:
        weight = np.asarray(arrs["weight"], dtype=float)
    else:
        weight = np.asarray(arrs[cfg["weight"]], dtype=float)
    df = pd.DataFrame({
        "t_raw": t_raw,
        "weight": weight,
        "true_t": np.asarray(arrs["true_t"], dtype=float) if "true_t" in arrs else np.nan,
        "reco_mass": np.asarray(arrs["reco_mass"], dtype=float),
        "sigma_epz": np.asarray(arrs["sigma_epz"], dtype=float),
        "reco_W": np.asarray(arrs["reco_W"], dtype=float),
    })
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["t_raw", "weight"])
    df = df[(df["t_raw"] > 0) & (df["weight"] > 0)]
    return df

def make_t_spectrum(t_values, weights, t_edges, lumi_invfb=0.0):
    t_values = np.asarray(t_values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    counts, _ = np.histogram(t_values, bins=t_edges, weights=weights)
    variances, _ = np.histogram(t_values, bins=t_edges, weights=weights * weights)
    widths = np.diff(t_edges)
    centers = 0.5 * (np.asarray(t_edges[:-1]) + np.asarray(t_edges[1:]))
    if lumi_invfb and lumi_invfb > 0:
        y = counts / lumi_invfb / widths
        yerr = np.sqrt(variances) / lumi_invfb / widths
        unit = "fb/GeV^2"
    else:
        y = counts / widths
        yerr = np.sqrt(variances) / widths
        unit = "yield/GeV^2"
    return pd.DataFrame({
        "t_low": t_edges[:-1],
        "t_high": t_edges[1:],
        "t_center": centers,
        "delta_center": np.sqrt(centers),
        "bin_width": widths,
        "sum_weight": counts,
        "sum_weight_err": np.sqrt(variances),
        "dsigma_dt": y,
        "dsigma_dt_err": yerr,
        "unit": unit,
    })

def prepare_amplitude_grid(spec, t_max, n_delta=2000, min_positive=0.0, low_delta_extrapolation="expfit"):
    delta_max = math.sqrt(t_max)
    delta_grid = np.linspace(0.0, delta_max, int(n_delta))
    delta_centers_all = np.asarray(spec["delta_center"], dtype=float)
    t_centers_all = np.asarray(spec["t_center"], dtype=float)
    dsdt_all = np.asarray(spec["dsigma_dt"], dtype=float)

    valid = np.isfinite(delta_centers_all) & np.isfinite(t_centers_all) & np.isfinite(dsdt_all) & (dsdt_all > min_positive)
    if np.sum(valid) < 2:
        raise RuntimeError("Too few positive dσ/dt bins for transform.")

    delta_centers = delta_centers_all[valid]
    t_centers = t_centers_all[valid]
    dsdt = dsdt_all[valid]
    amp_centers = np.sqrt(np.maximum(dsdt, 0.0))

    order = np.argsort(delta_centers)
    delta_centers = delta_centers[order]
    t_centers = t_centers[order]
    dsdt = dsdt[order]
    amp_centers = amp_centers[order]

    mode = low_delta_extrapolation.lower()
    if mode == "zero":
        x = delta_centers
        y = amp_centers
        left = 0.0
    elif mode == "constant":
        x = np.r_[0.0, delta_centers]
        y = np.r_[amp_centers[0], amp_centers]
        left = amp_centers[0]
    elif mode == "expfit":
        nfit = min(4, len(t_centers))
        coeff = np.polyfit(t_centers[:nfit], np.log(np.maximum(dsdt[:nfit], 1e-300)), deg=1)
        log_dsdt_0 = coeff[1]
        amp0 = math.sqrt(max(math.exp(log_dsdt_0), 0.0))
        if not np.isfinite(amp0) or amp0 <= 0:
            amp0 = amp_centers[0]
        x = np.r_[0.0, delta_centers]
        y = np.r_[amp0, amp_centers]
        left = amp0
    else:
        raise ValueError(f"Unknown low_delta_extrapolation={low_delta_extrapolation}")

    amp_grid = np.interp(delta_grid, x, y, left=left, right=0.0)
    return delta_grid, amp_grid

def hankel_transform_from_grid(delta_grid, amp_grid, b_fm_values):
    b_fm_values = np.asarray(b_fm_values, dtype=float)
    b_gevinv = np.abs(b_fm_values) / HBARC_GEV_FM
    F = np.zeros_like(b_gevinv, dtype=float)
    pref = delta_grid * amp_grid
    for i, b in enumerate(b_gevinv):
        F[i] = (1.0 / (2.0 * np.pi)) * np.trapz(pref * bessel_j0(b * delta_grid), delta_grid)
    return F

def normalize_profile(b_fm, F):
    b_fm = np.asarray(b_fm, dtype=float)
    F = np.asarray(F, dtype=float)
    area = np.trapz(F, b_fm)
    F_area = F / area if np.isfinite(area) and abs(area) > 0 else np.full_like(F, np.nan)
    maxabs = np.nanmax(np.abs(F)) if np.any(np.isfinite(F)) else np.nan
    F_max = F / maxabs if maxabs and np.isfinite(maxabs) and maxabs > 0 else np.full_like(F, np.nan)
    return F_area, F_max, area, maxabs

def transform_spectrum(spec, args):
    t_max = min(args.t_max, float(spec["t_high"].max()))
    delta_grid, amp_grid = prepare_amplitude_grid(spec, t_max=t_max, n_delta=args.n_delta, low_delta_extrapolation=args.low_delta_extrapolation)
    b_fm = np.linspace(args.b_min, args.b_max, args.n_b)
    F = hankel_transform_from_grid(delta_grid, amp_grid, b_fm)
    F_area, F_max, area, maxabs = normalize_profile(b_fm, F)
    prof = pd.DataFrame({"b_fm": b_fm, "b_GeV_inv": b_fm / HBARC_GEV_FM, "F_raw": F, "F_area_norm": F_area, "F_maxabs_norm": F_max})
    meta = {"t_max_used": t_max, "delta_max_used": math.sqrt(t_max), "area_raw_integral_over_b_fm": area, "max_abs_F_raw": maxabs}
    return prof, meta

def analytic_exponential_validation(args, outdir):
    B = args.validation_exp_slope
    A = 1.0
    delta_max = math.sqrt(args.validation_t_max)
    delta = np.linspace(0.0, delta_max, args.n_delta)
    amp = np.sqrt(A) * np.exp(-0.5 * B * delta * delta)
    b_fm = np.linspace(args.b_min, args.b_max, args.n_b)
    F_num = hankel_transform_from_grid(delta, amp, b_fm)
    _, F_num_max, _, _ = normalize_profile(b_fm, F_num)
    b_gevinv_abs = np.abs(b_fm) / HBARC_GEV_FM
    F_ana = np.exp(-(b_gevinv_abs * b_gevinv_abs) / (2.0 * B))
    _, F_ana_max, _, _ = normalize_profile(b_fm, F_ana)
    mask_abs = np.abs(b_fm) < min(args.b_max, 8.0)
    mask_frac = mask_abs & (np.abs(F_ana_max) > 1e-3)
    max_abs_diff = float(np.nanmax(np.abs(F_num_max[mask_abs] - F_ana_max[mask_abs])))
    if np.any(mask_frac):
        max_frac_diff = float(np.nanmax(np.abs(F_num_max[mask_frac] - F_ana_max[mask_frac]) / np.abs(F_ana_max[mask_frac])))
    else:
        max_frac_diff = np.nan
    val = pd.DataFrame({"b_fm": b_fm, "F_numeric_maxabs_norm": F_num_max, "F_analytic_maxabs_norm": F_ana_max})
    val.to_csv(os.path.join(outdir, "validation_exponential_profile.csv"), index=False)
    return {"validation": "exponential", "B_GeV_minus2": B, "t_max_GeV2": args.validation_t_max, "max_abs_diff_max_norm_core": max_abs_diff, "max_frac_diff_max_norm_core": max_frac_diff}

def toy_uncertainty(spec, args, rng):
    if args.n_toys <= 0:
        return None
    central = np.asarray(spec["dsigma_dt"], dtype=float)
    err = np.asarray(spec["dsigma_dt_err"], dtype=float)
    profiles = []
    last_prof = None
    for _ in range(args.n_toys):
        varied = np.clip(rng.normal(central, err), 0.0, None)
        toy_spec = spec.copy()
        toy_spec["dsigma_dt"] = varied
        try:
            prof, _ = transform_spectrum(toy_spec, args)
            last_prof = prof
            profiles.append(prof["F_area_norm"].to_numpy())
        except Exception:
            continue
    if len(profiles) == 0:
        return None
    arr = np.vstack(profiles)
    return pd.DataFrame({"b_fm": last_prof["b_fm"].to_numpy(), "F_area_norm_toy_p16": np.nanquantile(arr, 0.16, axis=0), "F_area_norm_toy_p50": np.nanquantile(arr, 0.50, axis=0), "F_area_norm_toy_p84": np.nanquantile(arr, 0.84, axis=0)})

def integration_convergence(spec, args):
    prof_nom, _ = transform_spectrum(spec, args)
    args_half = type("Args", (), vars(args))()
    args_half.n_delta = max(200, args.n_delta // 2)
    prof_half, _ = transform_spectrum(spec, args_half)
    diff = prof_nom["F_maxabs_norm"].to_numpy() - prof_half["F_maxabs_norm"].to_numpy()
    return {"validation": "integration_convergence", "n_delta_nominal": args.n_delta, "n_delta_half": args_half.n_delta, "max_abs_diff_F_maxabs_norm": float(np.nanmax(np.abs(diff))), "rms_diff_F_maxabs_norm": float(np.sqrt(np.nanmean(diff * diff)))}

def binning_stability(df, args):
    alt_edges = np.asarray(parse_float_list(args.alt_t_edges), dtype=float)
    nominal_edges = np.asarray(parse_float_list(args.t_edges), dtype=float)
    t = df["t_final"].to_numpy()
    w = df["weight"].to_numpy()
    spec_nom = make_t_spectrum(t, w, nominal_edges, lumi_invfb=args.lumi_invfb)
    spec_alt = make_t_spectrum(t, w, alt_edges, lumi_invfb=args.lumi_invfb)
    prof_nom, _ = transform_spectrum(spec_nom, args)
    prof_alt, _ = transform_spectrum(spec_alt, args)
    f_nom = prof_nom["F_maxabs_norm"].to_numpy()
    f_alt = prof_alt["F_maxabs_norm"].to_numpy()
    diff = f_nom - f_alt
    out = pd.DataFrame({"b_fm": prof_nom["b_fm"], "F_nominal_maxabs_norm": f_nom, "F_alt_binning_maxabs_norm": f_alt, "difference": diff})
    summary = {"validation": "binning_stability", "nominal_edges": args.t_edges, "alternative_edges": args.alt_t_edges, "max_abs_diff_F_maxabs_norm": float(np.nanmax(np.abs(diff))), "rms_diff_F_maxabs_norm": float(np.sqrt(np.nanmean(diff * diff)))}
    return out, summary

def plot_profiles(profile_dict, outpath, title="", ylabel="F(b) normalized"):
    if plt is None:
        return
    if HAS_MPLHEP:
        hep.style.use(hep.style.ROOT)
    fig, ax = plt.subplots(figsize=(8, 6))
    for label, df in profile_dict.items():
        ax.plot(df["b_fm"], df["F_area_norm"], linewidth=2, label=label)
    ax.set_xlabel(r"$b_T$ [fm]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def plot_t_spectrum(spec, outpath, title=""):
    if plt is None:
        return
    if HAS_MPLHEP:
        hep.style.use(hep.style.ROOT)
    fig, ax = plt.subplots(figsize=(8, 6))
    x = spec["t_center"].to_numpy()
    y = spec["dsigma_dt"].to_numpy()
    yerr = spec["dsigma_dt_err"].to_numpy()
    xerr = 0.5 * spec["bin_width"].to_numpy()
    ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="o", label="input spectrum")
    ax.set_yscale("log")
    ax.set_xlabel(r"$|t|$ [GeV$^2$]")
    ax.set_ylabel(r"$d\sigma/d|t|$ or $dN/d|t|$")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

# ==============================================================================
# JUPYTER NOTEBOOK WRAPPER
# ==============================================================================
def run_transform(tuple_path, tree_name="UpsilonTree", outdir="bT_transform_output",
                  method="eX", calibration="none", n_quantile_knots=12,
                  calibration_train_fraction=1.0, electron_energy=18.0,
                  sigma_epz_half_width=None, apply_sigma_epz=True, mass_min=8.0,
                  mass_max=12.0, w_min=20.0, use_corrected_proj_weight=True,
                  require_proton_tag_for_proj=True, lumi_invfb=10.0,
                  t_edges="0,0.25,0.35,0.50,0.75,1.00,1.50,2.00",
                  alt_t_edges="0,0.35,0.75,2.00", t_max=2.0, b_min=-10.0,
                  b_max=10.0, n_b=401, n_delta=4000, low_delta_extrapolation="expfit", 
                  n_toys=0, seed=12345, validation_exp_slope=20.0, validation_t_max=2.0):
    """
    Executes the entire transformation script natively within Python.
    """
    class Args: pass
    args = Args()
    
    args.tuple = tuple_path
    args.tree_name = tree_name
    args.outdir = outdir
    args.method = method
    args.calibration = calibration
    args.n_quantile_knots = n_quantile_knots
    args.calibration_train_fraction = calibration_train_fraction
    args.electron_energy = electron_energy
    args.apply_sigma_epz = apply_sigma_epz
    args.mass_min = mass_min
    args.mass_max = mass_max
    args.w_min = w_min
    args.use_corrected_proj_weight = use_corrected_proj_weight
    args.require_proton_tag_for_proj = require_proton_tag_for_proj
    args.lumi_invfb = lumi_invfb
    args.t_edges = t_edges
    args.alt_t_edges = alt_t_edges
    args.t_max = t_max
    args.b_min = b_min
    args.b_max = b_max
    args.n_b = n_b
    args.n_delta = n_delta
    args.low_delta_extrapolation = low_delta_extrapolation
    args.n_toys = n_toys
    args.seed = seed
    args.validation_exp_slope = validation_exp_slope
    args.validation_t_max = validation_t_max

    if sigma_epz_half_width is None:
        args.sigma_epz_half_width = 0.2 * args.electron_energy
    else:
        args.sigma_epz_half_width = sigma_epz_half_width

    args.outdir = os.path.expanduser(args.outdir)
    os.makedirs(args.outdir, exist_ok=True)
    t_edges_arr = np.asarray(parse_float_list(args.t_edges), dtype=float)
    if not np.all(np.diff(t_edges_arr) > 0):
        raise ValueError("t_edges must be strictly increasing.")

    print("=== Loading tuple ===")
    print(f"Input: {os.path.expanduser(args.tuple)}")
    print(f"Method: {args.method} ({METHODS[args.method]['label']})")
    print(f"Calibration: {args.calibration}")
    print(f"t edges: {args.t_edges}")

    df = load_tuple_dataframe(args.tuple, args.tree_name, args.method, args)
    t_cal, cal_curve = build_calibrated_reco_t(df, mode=args.calibration, n_knots=args.n_quantile_knots, seed=args.seed, train_fraction=args.calibration_train_fraction)
    df["t_final"] = t_cal
    df = df[(df["t_final"] > t_edges_arr[0]) & (df["t_final"] < t_edges_arr[-1])].copy()
    print(f"Selected entries in t range: {len(df)}")
    print(f"Sum weight in t range: {df['weight'].sum():.6g}")

    cal_curve.to_csv(os.path.join(args.outdir, "calibration_curve.csv"), index=False)
    df[["t_raw", "t_final", "true_t", "weight", "reco_mass", "sigma_epz", "reco_W"]].describe().to_csv(os.path.join(args.outdir, "input_dataframe_summary.csv"))

    spec = make_t_spectrum(df["t_final"].to_numpy(), df["weight"].to_numpy(), t_edges_arr, lumi_invfb=args.lumi_invfb)
    spec.to_csv(os.path.join(args.outdir, "t_spectrum_input.csv"), index=False)
    prof, meta = transform_spectrum(spec, args)
    prof.to_csv(os.path.join(args.outdir, "bT_profile.csv"), index=False)

    validation_rows = [analytic_exponential_validation(args, args.outdir), integration_convergence(spec, args)]
    binning_df, binning_summary = binning_stability(df, args)
    binning_df.to_csv(os.path.join(args.outdir, "validation_binning_stability_profile.csv"), index=False)
    validation_rows.append(binning_summary)

    truth_prof = None
    try:
        truth_df = df[(df["true_t"] > t_edges_arr[0]) & (df["true_t"] < t_edges_arr[-1])].copy()
        truth_spec = make_t_spectrum(truth_df["true_t"].to_numpy(), truth_df["weight"].to_numpy(), t_edges_arr, lumi_invfb=args.lumi_invfb)
        truth_spec.to_csv(os.path.join(args.outdir, "t_spectrum_truth_same_selection.csv"), index=False)
        truth_prof, _ = transform_spectrum(truth_spec, args)
        truth_prof.to_csv(os.path.join(args.outdir, "bT_profile_truth_same_selection.csv"), index=False)
        diff = prof["F_maxabs_norm"].to_numpy() - truth_prof["F_maxabs_norm"].to_numpy()
        validation_rows.append({"validation": "truth_vs_reco_profile_same_selection", "max_abs_diff_F_maxabs_norm": float(np.nanmax(np.abs(diff))), "rms_diff_F_maxabs_norm": float(np.sqrt(np.nanmean(diff * diff)))})
    except Exception as e:
        validation_rows.append({"validation": "truth_vs_reco_profile_same_selection", "error": str(e)})

    rng = np.random.default_rng(args.seed)
    toys = toy_uncertainty(spec, args, rng)
    if toys is not None:
        toys.to_csv(os.path.join(args.outdir, "bT_profile_toy_uncertainty.csv"), index=False)

    meta.update({
        "method": args.method,
        "method_label": METHODS[args.method]["label"],
        "calibration": args.calibration,
        "t_edges": args.t_edges,
        "low_delta_extrapolation": args.low_delta_extrapolation,
        "alt_t_edges": args.alt_t_edges,
        "lumi_invfb": args.lumi_invfb,
        "selection": {"mass_min": args.mass_min, "mass_max": args.mass_max, "w_min": args.w_min, "apply_sigma_epz": args.apply_sigma_epz, "sigma_epz_center": 2.0 * args.electron_energy, "sigma_epz_half_width": args.sigma_epz_half_width},
        "formula": "F(b) ∝ (1/(2π)) ∫ dΔ Δ J0(bΔ) sqrt(dσ/d|t|), Δ=sqrt(|t|)",
        "unit_note": "b is evaluated in GeV^-1 in the Bessel argument and written out as b_fm using hbar*c.",
    })
    with open(os.path.join(args.outdir, "transform_metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)
    pd.DataFrame(validation_rows).to_csv(os.path.join(args.outdir, "validation_summary.csv"), index=False)

    if plt is not None:
        plot_t_spectrum(spec, os.path.join(args.outdir, "t_spectrum_input.png"), title=f"{args.method}, {args.calibration}")
        profiles = {f"{args.method} {args.calibration}": prof}
        if truth_prof is not None:
            profiles["truth same selection"] = truth_prof
        plot_profiles(profiles, os.path.join(args.outdir, "bT_profile_comparison.png"), title=f"bT transform, {args.method}, {args.calibration}")
        val_exp = pd.read_csv(os.path.join(args.outdir, "validation_exponential_profile.csv"))
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot(val_exp["b_fm"], val_exp["F_numeric_maxabs_norm"], label="numeric")
        ax.plot(val_exp["b_fm"], val_exp["F_analytic_maxabs_norm"], linestyle="--", label="analytic")
        ax.set_xlabel(r"$b_T$ [fm]")
        ax.set_ylabel("max-normalized F(b)")
        ax.set_title("Exponential validation")
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(args.outdir, "validation_exponential_profile.png"), dpi=200)
        plt.close(fig)

    print("=== Done ===")
    print(f"Wrote outputs to: {args.outdir}")


In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

# Use standard ROOT style
hep.style.use(hep.style.ROOT)

def get_th1d_data(file_obj, hist_name):
    """Extracts bin contents, edges, and errors from a ROOT TH1D."""
    h = file_obj.Get(hist_name)
    if not h or not isinstance(h, ROOT.TH1):
        print(f"Warning: Could not retrieve TH1D '{hist_name}' from {file_obj.GetName()}")
        return None, None, None
        
    n_bins = h.GetNbinsX()
    
    # Extract edges (n_bins + 1)
    edges = np.array([h.GetBinLowEdge(i) for i in range(1, n_bins + 2)])
    # Extract contents and errors (n_bins)
    contents = np.array([h.GetBinContent(i) for i in range(1, n_bins + 1)])
    errors = np.array([h.GetBinError(i) for i in range(1, n_bins + 1)])
    
    return contents, edges, errors

def draw_epic_labels(ax):
    """Draws the standard ePIC performance block in the top left corner."""
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, 
            fontsize=14, fontweight='bold', ha='left', va='top')
    
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, 
            fontsize=13, ha='left', va='top')
    
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$", transform=ax.transAxes, 
            fontsize=13, ha='left', va='top')

def calc_pct_lost(c):
    """Calculates the percentage of events lost relative to the previous bin."""
    pct_lost = np.zeros_like(c, dtype=float)
    for i in range(1, len(c)):
        if c[i-1] > 0:
            pct_lost[i] = ((c[i-1] - c[i]) / c[i-1]) * 100.0
    return pct_lost

def main():
    # Input files
    file_10x100_path = "~/merged_elecs_test_10t100.root"
    file_18x275_path = "~/merged_elecs_test_tuples.root"
    
    f_10 = ROOT.TFile.Open(file_10x100_path, "READ")
    f_18 = ROOT.TFile.Open(file_18x275_path, "READ")
    
    if not f_10 or f_10.IsZombie() or not f_18 or f_18.IsZombie():
        print("Error opening ROOT files.")
        return

    hist_name = "h_CutFlow"
    
    # Create a figure with 2 subplots (Top: Absolute Cutflow, Bottom: % Lost)
    # Height ratios make the top plot 3x taller than the bottom plot
    fig, (ax_main, ax_loss) = plt.subplots(
        2, 1, figsize=(12, 10), 
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}, 
        sharex=True
    )
    
    fig.suptitle(r"$\Upsilon(nS) \rightarrow e^+e^-$ | Event Selection Cutflow", 
                 fontsize=22, fontweight='bold', y=0.95)
    
    # Pull the pre-binned data from the TH1Ds
    c_10, bins_10, err_10 = get_th1d_data(f_10, hist_name)
    c_18, bins_18, err_18 = get_th1d_data(f_18, hist_name)

    max_y = 0

    # --- TOP PANEL: Main Cutflow ---
    if c_10 is not None:
        hep.histplot(c_10, bins=bins_10, yerr=err_10, histtype='step', 
                     color='darkorange', label='10x100 GeV', ax=ax_main, lw=2)
        max_y = max(max_y, np.max(c_10))

    if c_18 is not None:
        hep.histplot(c_18, bins=bins_18, yerr=err_18, histtype='step', 
                     color='dodgerblue', label='18x275 GeV', ax=ax_main, lw=2)
        max_y = max(max_y, np.max(c_18))

    ax_main.set_ylabel("Remaining Events", fontsize=16)
    draw_epic_labels(ax_main)
    
    ax_main.set_yscale('log')
    ax_main.set_ylim(bottom=0.5)
    if max_y > 0: 
        ax_main.set_ylim(top=max_y * 200) # Ensure legend/text space
            
    ax_main.legend(loc="upper right", frameon=False, fontsize=14)


    # --- BOTTOM PANEL: Percentage Lost ---
    if c_10 is not None and c_18 is not None:
        pct_lost_10 = calc_pct_lost(c_10)
        pct_lost_18 = calc_pct_lost(c_18)
        
        # Plot the steps in the bottom panel
        hep.histplot(pct_lost_10, bins=bins_10, histtype='step', color='darkorange', ax=ax_loss, lw=2)
        hep.histplot(pct_lost_18, bins=bins_18, histtype='step', color='dodgerblue', ax=ax_loss, lw=2)
        
        # Add faint horizontal gridlines for readability (0%, 25%, 50%, 75%, 100%)
        for y_val in [0, 25, 50, 75, 100]:
            ax_loss.axhline(y_val, color='gray', linestyle='--', alpha=0.3, lw=1)

    ax_loss.set_ylabel("% Lost\n(per step)", fontsize=14)
    ax_loss.set_ylim(-5, 110) # Pad slightly below 0 and above 100

    # --- X-AXIS LABELS ---
    cut_names = [
        "1. Total Processed", 
        "2. Gen Upsilon Found", 
        "3. Gen Leps in Acc.",
        "4. Reco e+ Candidate", 
        "5. Reco e- Decay Cand", 
        "6. Reco Scat. e- Cand",
        "7. Mass Window [8, 12]", 
        r"8. $\Sigma(E-P_z)$ Exclusivity", 
        "9. W Phase Space Cut", 
        "10. Proton Tagged"
    ]
    
    # Apply labels only to the bottom axis
    ax_loss.set_xticks(np.arange(1, 11))
    ax_loss.set_xticklabels(cut_names, rotation=45, ha='right', fontsize=12)

    # Clean up layout
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig("Upsilon_EventSelection_Cutflow_with_Loss.png", dpi=300)
    print("Saved Upsilon_EventSelection_Cutflow_with_Loss.png")

    f_10.Close()
    f_18.Close()

if __name__ == "__main__":
    main()

In [ ]:
run_transform("~/merged_elecs_test_tuples.root", method="proj", outdir="~/my_transform_results")